In [ ]:
!nvidia-smi

Mon Sep 28 02:05:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/PROJECT'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

Mounted at /content/drive


In [ ]:
!pip install torch_geometric
!pip install slither-analyzer solc-select
!pip install shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.0/765.0 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.3/586.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 

In [ ]:
!ls -la "/content/drive/MyDrive/PROJECT/data/raw/"

total 7410
drwx------ 2 root root    4096 Aug 26 05:54 .
drwx------ 3 root root    4096 Aug 26 05:54 ..
-rw------- 1 root root 7579524 Aug 26 06:37 dataset_penelitian.zip


In [ ]:
!rm -rf /content/data/raw/dataset_penelitian

In [ ]:
!mkdir -p /content/data/raw
!unzip -q "/content/drive/MyDrive/PROJECT/data/raw/dataset_penelitian.zip" -d /content/data/raw/

In [ ]:
!find /content/data/raw/dataset_penelitian -maxdepth 2 -type d

/content/data/raw/dataset_penelitian
/content/data/raw/dataset_penelitian/0_normal
/content/data/raw/dataset_penelitian/2_wild_test
/content/data/raw/dataset_penelitian/3_expert_benchmark
/content/data/raw/dataset_penelitian/3_expert_benchmark/rentan
/content/data/raw/dataset_penelitian/3_expert_benchmark/aman
/content/data/raw/dataset_penelitian/1_reentrancy


In [ ]:
import os

for root in ['0_normal', '1_reentrancy', '3_expert_benchmark/aman', '3_expert_benchmark/rentan']:
    path = f'/content/data/raw/dataset_penelitian/{root}'
    count = len([f for f in os.listdir(path) if f.endswith('.sol')])
    print(f'{root}: {count} file .sol')

0_normal: 500 file .sol
1_reentrancy: 250 file .sol
3_expert_benchmark/aman: 35 file .sol
3_expert_benchmark/rentan: 31 file .sol


In [ ]:
import os, re
from collections import Counter

def get_pragma(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    match = re.search(r'pragma\s+solidity\s*([^;]+);', content)
    return match.group(1).strip() if match else "TIDAK_DITEMUKAN"

base = '/content/data/raw/dataset_penelitian'
folders = ['0_normal', '1_reentrancy', '3_expert_benchmark/aman', '3_expert_benchmark/rentan']

pragma_counter = Counter()
for folder in folders:
    path = os.path.join(base, folder)
    for fname in os.listdir(path):
        if fname.endswith('.sol'):
            pragma_counter[get_pragma(os.path.join(path, fname))] += 1

for pragma, count in pragma_counter.most_common(20):
    print(f'{count:>4}  {pragma}')

 101  0.6.12
  55  0.5.17
  45  0.6.5
  43  0.6.10
  41  ^0.8.0
  36  0.4.21
  31  0.6.6
  27  ^0.5.0
  25  0.7.0
  18  0.5.16
  17  ^0.4.19
  16  0.4.24
  15  ^0.4.24
  15  ^0.6.12
  14  0.6.8
  14  ^0.8.24
  13  0.7.4
  12  ^0.4.21
  12  ^0.8.20
  11  ^0.8.4


In [ ]:
import re

def resolve_solc_version(pragma_str):
    match = re.search(r'(\d+\.\d+\.\d+)', pragma_str)
    return match.group(1) if match else None

In [ ]:
unique_versions = set()
no_pragma_count = 0

for pragma, count in pragma_counter.items():
    v = resolve_solc_version(pragma)
    if v:
        unique_versions.add(v)
    else:
        no_pragma_count += count

print(f"Total versi unik yang perlu diinstall: {len(unique_versions)}")
print(f"Kontrak tanpa pragma terdeteksi: {no_pragma_count}")
print(sorted(unique_versions))

Total versi unik yang perlu diinstall: 68
Kontrak tanpa pragma terdeteksi: 1
['0.4.0', '0.4.10', '0.4.11', '0.4.13', '0.4.14', '0.4.15', '0.4.16', '0.4.17', '0.4.18', '0.4.19', '0.4.2', '0.4.20', '0.4.21', '0.4.22', '0.4.23', '0.4.24', '0.4.25', '0.4.26', '0.4.4', '0.4.8', '0.5.0', '0.5.10', '0.5.11', '0.5.12', '0.5.15', '0.5.16', '0.5.17', '0.5.2', '0.5.4', '0.5.5', '0.5.6', '0.5.7', '0.5.8', '0.5.9', '0.6.0', '0.6.10', '0.6.11', '0.6.12', '0.6.2', '0.6.3', '0.6.4', '0.6.5', '0.6.6', '0.6.7', '0.6.8', '0.6.9', '0.7.0', '0.7.1', '0.7.2', '0.7.3', '0.7.4', '0.7.5', '0.7.6', '0.8.0', '0.8.1', '0.8.10', '0.8.2', '0.8.20', '0.8.21', '0.8.22', '0.8.24', '0.8.26', '0.8.27', '0.8.3', '0.8.4', '0.8.6', '0.8.7', '0.8.9']


In [ ]:
import os
os.environ['SOLC_SELECT_INSTALL_DIR'] = '/content/drive/MyDrive/PROJECT/solc_binaries'
os.makedirs(os.environ['SOLC_SELECT_INSTALL_DIR'], exist_ok=True)

from solc_select import solc_select
installed = solc_select.installed_versions()
gagal = []
for v in sorted(unique_versions):
    if v in installed:
        continue
    try:
        solc_select.install_artifacts([v])
        print(f'OK   {v}')
    except Exception as e:
        gagal.append(v)
        print(f'GAGAL {v}: {e}')
print('Versi gagal install:', gagal)

Installing solc '0.4.0'...
Version '0.4.0' installed.
OK   0.4.0
Installing solc '0.4.10'...
Version '0.4.10' installed.
OK   0.4.10
Installing solc '0.4.11'...
Version '0.4.11' installed.
OK   0.4.11
Installing solc '0.4.13'...
Version '0.4.13' installed.
OK   0.4.13
Installing solc '0.4.14'...
Version '0.4.14' installed.
OK   0.4.14
Installing solc '0.4.15'...
Version '0.4.15' installed.
OK   0.4.15
Installing solc '0.4.16'...
Version '0.4.16' installed.
OK   0.4.16
Installing solc '0.4.17'...
Version '0.4.17' installed.
OK   0.4.17
Installing solc '0.4.18'...
Version '0.4.18' installed.
OK   0.4.18
Installing solc '0.4.19'...
Version '0.4.19' installed.
OK   0.4.19
Installing solc '0.4.2'...
Version '0.4.2' installed.
OK   0.4.2
Installing solc '0.4.20'...
Version '0.4.20' installed.
OK   0.4.20
Installing solc '0.4.21'...
Version '0.4.21' installed.
OK   0.4.21
Installing solc '0.4.22'...
Version '0.4.22' installed.
OK   0.4.22
Installing solc '0.4.23'...
Version '0.4.23' installed

In [ ]:
import os
p = '/content/drive/MyDrive/PROJECT/data/processed/manifest.csv'
print(os.path.exists(p))

False


In [ ]:
import os, pathlib
DRIVE_SOLC = '/content/drive/MyDrive/PROJECT/solc_binaries'
os.makedirs(DRIVE_SOLC, exist_ok=True)
link = pathlib.Path.home() / '.solc-select'
if not link.exists():
    os.symlink(DRIVE_SOLC, link)
!solc-select versions

0.4.0
0.4.2
0.4.4
0.4.8
0.4.10
0.4.11
0.4.13
0.4.14
0.4.15
0.4.16
0.4.17
0.4.18
0.4.19
0.4.20
0.4.21
0.4.22
0.4.23
0.4.24
0.4.25
0.4.26
0.5.0
0.5.2
0.5.4
0.5.5
0.5.6
0.5.7
0.5.8
0.5.9
0.5.10
0.5.11
0.5.12
0.5.15
0.5.16
0.5.17
0.6.0 (current, set by /root/.solc-select/global-version)
0.6.2
0.6.3
0.6.4
0.6.5
0.6.6
0.6.7
0.6.8
0.6.9
0.6.10
0.6.11
0.6.12
0.7.0
0.7.1
0.7.2
0.7.3
0.7.4
0.7.5
0.7.6
0.8.0
0.8.1
0.8.2
0.8.3
0.8.4
0.8.6
0.8.7
0.8.9
0.8.10
0.8.20
0.8.21
0.8.22
0.8.24
0.8.26
0.8.27


In [ ]:
import os, csv, subprocess
from slither import Slither

os.chdir('/content')

BASE = '/content/data/raw/dataset_penelitian'
MANIFEST = '/content/drive/MyDrive/PROJECT/data/processed/manifest.csv'
os.makedirs(os.path.dirname(MANIFEST), exist_ok=True)

folders = {
    '0_normal': (0, 'train'),
    '1_reentrancy': (1, 'train'),
    '3_expert_benchmark/aman': (0, 'benchmark'),
    '3_expert_benchmark/rentan': (1, 'benchmark'),
}

# kumpulkan daftar kerja, lalu urutkan berdasarkan versi solc
tugas = []
for folder, (label, split) in folders.items():
    path = os.path.join(BASE, folder)
    for fname in sorted(os.listdir(path)):
        if not fname.endswith('.sol'):
            continue
        fp = os.path.join(path, fname)
        pragma = get_pragma(fp)
        tugas.append((resolve_solc_version(pragma) or 'ZZZ', fp, fname, folder, label, split, pragma))
tugas.sort()

# lanjutkan dari hasil sebelumnya kalau ada
sudah = set()
if os.path.exists(MANIFEST):
    with open(MANIFEST) as f:
        sudah = {r['path'] for r in csv.DictReader(f)}
    print(f'Melanjutkan, {len(sudah)} kontrak sudah diproses')

header = ['path','nama_file','folder','label','split','pragma','solc','status','alasan','jumlah_fungsi','jumlah_node']
baru = not os.path.exists(MANIFEST)
f = open(MANIFEST, 'a', newline='')
w = csv.writer(f)
if baru:
    w.writerow(header)

versi_aktif = None
for i, (ver, fp, fname, folder, label, split, pragma) in enumerate(tugas, 1):
    if fp in sudah:
        continue
    status, alasan, n_fn, n_node = 'GAGAL', '', 0, 0
    if ver == 'ZZZ':
        alasan = 'pragma_tidak_ditemukan'
    else:
        try:
            if ver != versi_aktif:
                subprocess.run(['solc-select', 'use', ver], capture_output=True, check=True)
                versi_aktif = ver
            sl = Slither(fp)
            n_fn = sum(len(c.functions) for c in sl.contracts)
            n_node = sum(len(fn.nodes) for c in sl.contracts for fn in c.functions)
            status = 'OK' if n_fn > 0 else 'GAGAL'
            if n_fn == 0:
                alasan = 'tidak_ada_fungsi'
        except Exception as e:
            alasan = str(e).replace('\n', ' ')[:200]
    w.writerow([fp, fname, folder, label, split, pragma, ver if ver != 'ZZZ' else '', status, alasan, n_fn, n_node])
    f.flush()
    print(f'[{i}/{len(tugas)}] {status} {ver} {fname} {alasan[:60]}')

f.close()
print('Selesai, manifest tersimpan di', MANIFEST)

[1/816] OK 0.4.0 etherbank.sol 


[2/816] OK 0.4.10 0x01805590991328db7f1ef0e8e4b071ad8f58c8ce.sol 


[3/816] OK 0.4.10 0x0000000000b3F879cb30FE243b4Dfee438691c04.sol 
[4/816] OK 0.4.10 etherstore.sol 


[5/816] GAGAL 0.4.11 0x0006157838d5a6b33ab66588a6a693a57c869999.sol tidak_ada_fungsi


[6/816] OK 0.4.11 0x001ea8150f4965195e10e5b5568047e1555a6dcd.sol 


[7/816] OK 0.4.11 0x01417fea37677b581c1237b74a86c225136b3465.sol 


... (758 outputs left out)

[811/816] OK 0.8.9 0x005aefa801f03d7cd56d577bf91c9f4ca6c3c528.sol 


[812/816] OK 0.8.9 0x005d1123878fc55fbd56b54c73963b234a64af3c.sol 


[813/816] OK 0.8.9 0x005f8205ddcbb28b1295d5c28b152214197b32f8.sol 


[814/816] OK 0.8.9 0x0060228f198ec12089053786e762137d7cc70c60.sol 
[815/816] OK 0.8.9 0x007329768f4462c82d28d85b88e0fd6945847ac5.sol 
[816/816] GAGAL ZZZ 0x000000007a58f5f58e697e51ab0357bc9e260a04.sol pragma_tidak_ditemukan
Selesai, manifest tersimpan di /content/drive/MyDrive/PROJECT/data/processed/manifest.csv


In [ ]:
import pandas as pd
MANIFEST = '/content/drive/MyDrive/PROJECT/data/processed/manifest.csv'
df = pd.read_csv(MANIFEST)

print(f'Total baris: {len(df)} (harusnya 816)')
print(f'Duplikat path: {df.path.duplicated().sum()}')
print()
print(pd.crosstab([df.split, df.folder], df.status))
print()
print('Alasan gagal:')
print(df[df.status == 'GAGAL'].alasan.value_counts().head(10))
print()
ok = df[df.status == 'OK']
print('Statistik ukuran CFG kontrak yang lolos:')
print(ok[['jumlah_fungsi', 'jumlah_node']].describe())

Total baris: 816 (harusnya 816)
Duplikat path: 0

status                               GAGAL   OK
split     folder                               
benchmark 3_expert_benchmark/aman       33    2
          3_expert_benchmark/rentan      2   29
train     0_normal                      20  480
          1_reentrancy                  79  171

Alasan gagal:
alasan
Invalid compilation:  Invalid solc compilation Error: Expected pragma, import directive or contract/interface/library/struct/enum definition.     --> /content/data/raw/dataset_penelitian/1_reentrancy/    40
Invalid compilation:  Invalid solc compilation Warning: SPDX license identifier not provided in source file. Before publishing, consider adding a comment containing "SPDX-License-Identifier: <SPDX-Lic    16
Invalid compilation:  Invalid solc compilation Exception during compilation: /solidity/libyul/backends/evm/AsmCodeGen.cpp(210): Throw in function static void yul::CodeGenerator::assemble(const yul::Bl     7
Invalid compilation

In [ ]:
gagal = df[df.status == 'GAGAL'].copy()

def kategori(a):
    a = str(a)
    if 'not found' in a: return 'impor_hilang'
    if 'requires different compiler' in a: return 'versi_salah'
    if 'Expected pragma' in a: return 'sintaks_tidak_valid'
    if 'tidak_ada_fungsi' in a: return 'interface_saja'
    return 'lainnya'

gagal['kategori'] = gagal.alasan.map(kategori)
print(pd.crosstab(gagal.folder, gagal.kategori))

kategori                   impor_hilang  interface_saja  lainnya  \
folder                                                             
0_normal                              0               1        6   
1_reentrancy                          0               0       25   
3_expert_benchmark/aman              33               0        0   
3_expert_benchmark/rentan             0               0        2   

kategori                   sintaks_tidak_valid  versi_salah  
folder                                                       
0_normal                                     2           11  
1_reentrancy                                49            5  
3_expert_benchmark/aman                      0            0  
3_expert_benchmark/rentan                    0            0  


In [ ]:
sub = gagal[gagal.kategori == 'lainnya'].alasan.fillna('').astype(str)
for a in sub.head(8):
    print(a[:250], '\n')

Invalid compilation:  Invalid solc compilation /content/data/raw/dataset_penelitian/0_normal/0x06b7dbc16e445054386b01894eba31819b71d261.sol:530:5: Error: Expected identifier, got 'RParen' alue)�ɹ�ִ��ʱ 

Invalid compilation:  Invalid solc compilation /content/data/raw/dataset_penelitian/0_normal/0x270CA91dB3866bd9924502A6e4De33f7a6BeaAe9.sol:15:1: Error: Identifier already declared. contract ERC20Basi 

 

 

Invalid compilation:  Invalid solc compilation /content/data/raw/dataset_penelitian/0_normal/0x748e57656d093aa1bbcde2900092a87110c1c079.sol:530:5: Error: Expected identifier but got ')' alue)�ɹ�ִ��ʱ�� 

Invalid compilation:  Invalid solc compilation /content/data/raw/dataset_penelitian/0_normal/0xbb1416998df32605ebce0348a6c87aa9131be9a9.sol:530:5: Error: Expected identifier but got ')' alue)�ɹ�ִ��ʱ�� 

Invalid compilation:  Invalid solc compilation /content/data/raw/dataset_penelitian/1_reentrancy/0x003b62aa2d510e983a46b3b3308664faeaaaed2e.sol:323:42: Error: Expected ';' but got '{

In [ ]:
import re
pola = sub.map(lambda a: re.sub(r'/[^\s]+', '<path>', a)[:110])
print(pola.value_counts().head(10))

alasan
Invalid compilation:  Invalid solc compilation Warning: SPDX license identifier not provided in source file. B    16
Invalid compilation:  Invalid solc compilation Exception during compilation: <path> Throw in function static v     7
Invalid compilation:  Invalid solc compilation <path> Error: Expected identifier but got ')' alue)�ɹ�ִ��ʱ��        2
                                                                                                                   2
Invalid compilation:  Invalid solc compilation <path> Error: Identifier already declared. contract ERC20Basi       1
Invalid compilation:  Invalid solc compilation <path> Error: Expected identifier, got 'RParen' alue)�ɹ�ִ��ʱ        1
Invalid compilation:  Invalid solc compilation <path> Error: Expected ';' but got '{'         (bool su             1
Invalid compilation:  Invalid solc compilation <path> Error: Identifier already declared. contract Cont            1
Invalid compilation:  Invalid solc compilation Error: Ide

In [ ]:
from packaging.specifiers import SpecifierSet
from packaging.version import Version
from solc_select import solc_select
import re

terpasang = sorted([Version(v) for v in solc_select.installed_versions()])

def resolve_solc_version_v2(pragma_str):
    if not pragma_str or pragma_str == 'TIDAK_DITEMUKAN':
        return None
    s = str(pragma_str).replace('^', '~=').strip()
    try:
        spec = SpecifierSet(','.join(p.strip() for p in s.split() if p.strip()))
        cocok = [v for v in terpasang if v in spec]
        if cocok:
            return str(max(cocok))
    except Exception:
        pass
    m = re.search(r'(\d+\.\d+\.\d+)', str(pragma_str))
    return m.group(1) if m else None

In [ ]:
uji = gagal[gagal.kategori.isin(['sintaks_tidak_valid', 'versi_salah'])].head(10)
for _, r in uji.iterrows():
    print(f'{r.pragma:25} lama: {r.solc:8} baru: {resolve_solc_version_v2(r.pragma)}')

^0.4.16                   lama: 0.4.16   baru: 0.4.26
^0.4.18                   lama: 0.4.18   baru: 0.4.26
^0.5.0                    lama: 0.5.0    baru: 0.5.17
>=0.5.0 <0.7.0            lama: 0.5.0    baru: 0.6.12
^0.5.0                    lama: 0.5.0    baru: 0.5.17
>=0.6.0 <0.8.0            lama: 0.6.0    baru: 0.7.6
^0.6.0                    lama: 0.6.0    baru: 0.6.12
^0.6.0                    lama: 0.6.0    baru: 0.6.12
>=0.6.0 <0.8.0            lama: 0.6.0    baru: 0.7.6
^0.6.0                    lama: 0.6.0    baru: 0.6.12


jalankan ulang, tapi khusus kontrak berstatus GAGAL saja. Yang 682 sudah OK tidak disentuh.

In [ ]:
import os, csv, subprocess
from slither import Slither
import pandas as pd

os.chdir('/content')
MANIFEST = '/content/drive/MyDrive/PROJECT/data/processed/manifest.csv'
df = pd.read_csv(MANIFEST)

# hanya kontrak yang berpeluang pulih, file rusak dan impor hilang dilewati
target = df[(df.status == 'GAGAL') & (~df.alasan.fillna('').str.contains('not found'))].copy()
target['solc_baru'] = target.pragma.map(resolve_solc_version_v2)
target = target.sort_values('solc_baru')
print(f'{len(target)} kontrak akan dicoba ulang')

hasil = {}
versi_aktif = None
for i, (_, r) in enumerate(target.iterrows(), 1):
    ver = r.solc_baru
    status, alasan, n_fn, n_node = 'GAGAL', '', 0, 0
    if not ver:
        alasan = 'pragma_tidak_ditemukan'
    else:
        try:
            if ver != versi_aktif:
                subprocess.run(['solc-select', 'use', ver], capture_output=True, check=True)
                versi_aktif = ver
            sl = Slither(r.path)
            n_fn = sum(len(c.functions) for c in sl.contracts)
            n_node = sum(len(fn.nodes) for c in sl.contracts for fn in c.functions)
            status = 'OK' if n_fn > 0 else 'GAGAL'
            if n_fn == 0:
                alasan = 'tidak_ada_fungsi'
        except Exception as e:
            alasan = str(e).replace('\n', ' ')[:200]
    hasil[r.path] = (ver, status, alasan, n_fn, n_node)
    print(f'[{i}/{len(target)}] {status} {ver} {r.nama_file} {alasan[:50]}')

101 kontrak akan dicoba ulang


[1/101] GAGAL 0.4.19 0x06b7dbc16e445054386b01894eba31819b71d261.sol Invalid compilation:  Invalid solc compilation /co


[2/101] GAGAL 0.4.24 0x270CA91dB3866bd9924502A6e4De33f7a6BeaAe9.sol Invalid compilation:  Invalid solc compilation /co


[3/101] GAGAL 0.4.26 0x0006157838d5a6b33ab66588a6a693a57c869999.sol tidak_ada_fungsi


[4/101] OK 0.4.26 0x001575786dfa7b9d9d1324ec308785738f80a951.sol 


[5/101] OK 0.4.26 0x0014966c7439d8c10aaf0204036198236d9b0912.sol 


... (87 outputs left out)

[97/101] OK 0.8.27 0x006277d659707f787fe90eed22f1d2e5c71af4ab.sol 


[98/101] OK 0.8.27 0x00A0a01fef5A3210ea387BA725F0F9cA91BfE4DD.sol 


[99/101] OK 0.8.27 0x01385c810e2ec176bf065d5179e76dbad2a01173.sol 


[100/101] OK 0.8.27 0x002f0efb31696670cb3d78b04df2f41f88c73125.sol 
[101/101] OK 0.8.27 0x000000007a58f5f58e697e51ab0357bc9e260a04.sol 


In [ ]:
for path, (ver, status, alasan, n_fn, n_node) in hasil.items():
    if status == 'OK':
        m = df.path == path
        df.loc[m, ['solc', 'status', 'alasan', 'jumlah_fungsi', 'jumlah_node']] = [ver, status, '', n_fn, n_node]

MANIFEST2 = '/content/drive/MyDrive/PROJECT/data/processed/manifest_v2.csv'
df.to_csv(MANIFEST2, index=False)

print(pd.crosstab([df.split, df.folder], df.status))
pulih = sum(1 for v in hasil.values() if v[1] == 'OK')
print(f'\nBerhasil dipulihkan: {pulih} dari {len(target)} kontrak')

status                               GAGAL   OK
split     folder                               
benchmark 3_expert_benchmark/aman       33    2
          3_expert_benchmark/rentan      2   29
train     0_normal                      12  488
          1_reentrancy                  74  176

Berhasil dipulihkan: 13 dari 101 kontrak


In [ ]:
sisa = df[(df.status == 'GAGAL') & (~df.alasan.fillna('').str.contains('not found'))].copy()
sisa['kategori'] = sisa.alasan.fillna('').map(kategori)
print(pd.crosstab(sisa.folder, sisa.kategori))
print()
print(sisa.alasan.fillna('').map(lambda a: re.sub(r'/[^\s]+', '<path>', str(a))[:100]).value_counts().head(8))

kategori                   interface_saja  lainnya  sintaks_tidak_valid  \
folder                                                                    
0_normal                                1        5                    2   
1_reentrancy                            0       25                   49   
3_expert_benchmark/rentan               0        2                    0   

kategori                   versi_salah  
folder                                  
0_normal                             4  
1_reentrancy                         0  
3_expert_benchmark/rentan            0  

alasan
Invalid compilation:  Invalid solc compilation Error: Expected pragma, import directive or contract<    51
Invalid compilation:  Invalid solc compilation Warning: SPDX license identifier not provided in sour    16
Invalid compilation:  Invalid solc compilation Exception during compilation: <path> Throw in functio     7
Invalid compilation:  Invalid solc compilation Error: Source file requires different compi

In [ ]:
import re

pola51 = sisa[sisa.alasan.fillna('').str.contains('Expected pragma, import directive')]
print(f'{len(pola51)} kontrak dengan pola ini\n')

for _, r in pola51.head(4).iterrows():
    m = re.search(r'\.sol:(\d+):(\d+)', str(r.alasan))
    baris = int(m.group(1)) if m else 1
    print('=' * 70)
    print(r.nama_file, '| pragma:', r.pragma, '| baris error:', baris)
    with open(r.path, encoding='utf-8', errors='replace') as f:
        isi = f.readlines()
    awal = max(0, baris - 4)
    for i in range(awal, min(len(isi), baris + 3)):
        tanda = '>>>' if i == baris - 1 else '   '
        print(f'{tanda} {i+1:5} | {isi[i].rstrip()[:110]}')

51 kontrak dengan pola ini

0x3b4cAAAF6F3ce5Bee2871C89987cbd825Ac30822.sol | pragma: 0.6.12 | baris error: 1
>>>     1 | pragma solidity 0.6.12;
        2 | 
        3 | library Address {
        4 | 
0x0a98ad316dfbcdaef91e382348a0c3639ec8f9d3.sol | pragma: 0.6.12 | baris error: 1
>>>     1 | pragma solidity 0.6.12;
        2 | 
        3 | contract Initializable {
        4 | 
0x0b07234f0440f322a6afc0707383391b16ce8dda.sol | pragma: 0.6.12 | baris error: 1
>>>     1 | pragma solidity 0.6.12;
        2 | 
        3 | contract Initializable {
        4 | 
0x0cfb151de2c34aceb532f43683e5b7bed62f298f.sol | pragma: 0.6.12 | baris error: 1
>>>     1 | pragma solidity 0.6.12;
        2 | pragma experimental ABIEncoderV2;
        3 | 
        4 | contract IAbacusOracle{


In [ ]:
for _, r in pola51.head(5).iterrows():
    with open(r.path, 'rb') as f:
        b = f.read(60)
    print(r.nama_file[:34])
    print('   byte awal :', b[:30])
    print('   ada BOM   :', b.startswith(b'\xef\xbb\xbf'))
    print('   ada CRLF  :', b'\r\n' in b)

0x3b4cAAAF6F3ce5Bee2871C89987cbd82
   byte awal : b'pragma solidity 0.6.12;\n\nlibra'
   ada BOM   : False
   ada CRLF  : False
0x0a98ad316dfbcdaef91e382348a0c363
   byte awal : b'pragma solidity 0.6.12;\n\ncontr'
   ada BOM   : False
   ada CRLF  : False
0x0b07234f0440f322a6afc0707383391b
   byte awal : b'pragma solidity 0.6.12;\n\ncontr'
   ada BOM   : False
   ada CRLF  : False
0x0cfb151de2c34aceb532f43683e5b7be
   byte awal : b'pragma solidity 0.6.12;\npragma'
   ada BOM   : False
   ada CRLF  : False
0x0e68432827674ad048b803d1ee289ae7
   byte awal : b'pragma solidity 0.6.12;\npragma'
   ada BOM   : False
   ada CRLF  : False


In [ ]:
fp = pola51.iloc[0].path
!solc-select use 0.6.12
!solc --version
!solc "{fp}" --ast-compact-json 2>&1 | head -20

Switched global version to 0.6.12
solc, the solidity compiler commandline interface
Version: 0.6.12+commit.27d51765.Linux.g++
Error: Expected pragma, import directive or contract/interface/library/struct/enum definition.
    --> /content/data/raw/dataset_penelitian/0_normal/0x3b4cAAAF6F3ce5Bee2871C89987cbd825Ac30822.sol:2343:1:
     |
2343 | MINTER_ROLE");
     | ^^^^^^^^^^^



In [ ]:
fp = pola51.iloc[0].path
with open(fp, encoding='utf-8', errors='replace') as f:
    isi = f.readlines()
for i in range(2336, 2346):
    print(f'{i+1:5} | {repr(isi[i])}')

 2337 | '        require(!paused(), "ERC20Pausable: token transfer while paused");\n'
 2338 | '\n'
 2339 | '    }\n'
 2340 | '\n'
 2341 | '}\n'
 2342 | '\n'
 2343 | 'MINTER_ROLE");\n'
 2344 | '\n'
 2345 | '    bytes32 public constant BURNER_ROLE = keccak256("BURNER_ROLE");\n'
 2346 | '\n'


In [ ]:
def cacat(path):
    with open(path, encoding='utf-8', errors='replace') as f:
        baris = f.readlines()
    hasil = []
    for i, b in enumerate(baris, 1):
        tanpa_komentar = b.split('//')[0]
        if tanpa_komentar.count('"') % 2 == 1:
            hasil.append((i, b.rstrip()[:70]))
    return hasil

total_kena = 0
for _, r in pola51.iterrows():
    c = cacat(r.path)
    if c:
        total_kena += 1
print(f'{total_kena} dari {len(pola51)} kontrak punya string literal terpotong')

# contoh isinya
for _, r in pola51.head(3).iterrows():
    print('\n', r.nama_file[:34])
    for ln, txt in cacat(r.path)[:4]:
        print(f'   baris {ln}: {repr(txt)}')

5 dari 51 kontrak punya string literal terpotong

 0x3b4cAAAF6F3ce5Bee2871C89987cbd82
   baris 2343: 'MINTER_ROLE");'

 0x0a98ad316dfbcdaef91e382348a0c363

 0x0b07234f0440f322a6afc0707383391b


In [ ]:
import re
info = []
for _, r in pola51.iterrows():
    m = re.search(r'(\d+):(\d+):', str(r.alasan))
    if not m:
        continue
    ln = int(m.group(1))
    with open(r.path, encoding='utf-8', errors='replace') as f:
        isi = f.readlines()
    teks = isi[ln-1].rstrip() if ln <= len(isi) else '<di luar file>'
    info.append((r.nama_file[:20], ln, len(isi), teks[:60]))

print(f'{len(info)} dari {len(pola51)} berhasil diparse\n')
for nama, ln, total, teks in info[:15]:
    print(f'{nama} | baris {ln}/{total} | {repr(teks)}')

0 dari 51 berhasil diparse



In [ ]:
import re, subprocess

versi_aktif = None
temuan = []
for _, r in pola51.head(12).iterrows():
    ver = resolve_solc_version_v2(r.pragma)
    if ver != versi_aktif:
        subprocess.run(['solc-select', 'use', ver], capture_output=True)
        versi_aktif = ver
    out = subprocess.run(['solc', r.path], capture_output=True, text=True)
    pesan = out.stderr + out.stdout
    m = re.search(r'\.sol:(\d+):(\d+)', pesan)
    if not m:
        print(r.nama_file[:24], '| tidak ada nomor baris')
        continue
    ln = int(m.group(1))
    with open(r.path, encoding='utf-8', errors='replace') as f:
        isi = f.readlines()
    teks = isi[ln-1].rstrip() if ln <= len(isi) else '<di luar file>'
    sebelum = isi[ln-2].rstrip() if ln >= 2 else ''
    temuan.append((r.nama_file[:24], ln, len(isi), sebelum[:50], teks[:60]))

for nama, ln, total, sebelum, teks in temuan:
    print(f'{nama} | baris {ln}/{total}')
    print(f'   sebelum : {repr(sebelum)}')
    print(f'   error   : {repr(teks)}')

0x3b4cAAAF6F3ce5Bee2871C | baris 2343/2591
   sebelum : ''
   error   : 'MINTER_ROLE");'
0x0a98ad316dfbcdaef91e38 | baris 1247/4730
   sebelum : ''
   error   : 'c'
0x0b07234f0440f322a6afc0 | baris 1138/4388
   sebelum : ''
   error   : 'c'
0x0cfb151de2c34aceb532f4 | baris 3825/6786
   sebelum : ''
   error   : 'ublic orderUSDValue;'
0x0e68432827674ad048b803 | baris 3825/6570
   sebelum : ''
   error   : 'ucts and Initializer================'
0x116b1db5e116b7b9b5e2cd | baris 3825/6495
   sebelum : ''
   error   : 'ucts and Initializer================'
0x132ab3156a97b321b547c7 | baris 1138/4292
   sebelum : ''
   error   : 'c'
0x1ea761e7b0e4e4afbe1b79 | baris 3825/6492
   sebelum : ''
   error   : 'ucts and Initializer================'
0x412162b71a193caaff8d6f | baris 1138/4292
   sebelum : ''
   error   : 'c'
0x41517D4102Cc6ddB4f6daC | baris 1138/4388
   sebelum : ''
   error   : 'c'
0x4ebbce881f5f233c4bf683 | baris 1138/4388
   sebelum : ''
   error   : 'c'
0x52d70788b6bf2729060e11 | 

In [ ]:
rusak = df.status == 'GAGAL'
df.loc[rusak & df.alasan.fillna('').str.contains('Expected pragma, import directive'), 'alasan'] = 'file_rusak_sumber'
df.loc[rusak & df.alasan.fillna('').str.contains('not found'), 'alasan'] = 'impor_tidak_lengkap'
df.to_csv(MANIFEST2, index=False)
print(df[df.status=='GAGAL'].alasan.value_counts())

alasan
file_rusak_sumber                                                                                                                                                                                           51
impor_tidak_lengkap                                                                                                                                                                                         33
Invalid compilation:  Invalid solc compilation Warning: SPDX license identifier not provided in source file. Before publishing, consider adding a comment containing "SPDX-License-Identifier: <SPDX-Lic    16
Invalid compilation:  Invalid solc compilation Exception during compilation: /solidity/libyul/backends/evm/AsmCodeGen.cpp(210): Throw in function static void yul::CodeGenerator::assemble(const yul::Bl     7
Invalid compilation:  Invalid solc compilation Error: Source file requires different compiler version (current compiler is 0.6.0+commit.26b70077.Linux.g++ - note tha

In [ ]:
for _, r in pola51.head(6).iterrows():
    with open(r.path, 'rb') as f:
        b = f.read()
    cr_sendiri = b.count(b'\r') - b.count(b'\r\n')
    print(f'{r.nama_file[:26]} | CR sendirian: {cr_sendiri} | NUL: {b.count(bytes([0]))} | ukuran: {len(b)}')

0x3b4cAAAF6F3ce5Bee2871C89 | CR sendirian: 0 | NUL: 0 | ukuran: 44903
0x0a98ad316dfbcdaef91e3823 | CR sendirian: 0 | NUL: 0 | ukuran: 64135
0x0b07234f0440f322a6afc070 | CR sendirian: 0 | NUL: 0 | ukuran: 59040
0x0cfb151de2c34aceb532f436 | CR sendirian: 0 | NUL: 0 | ukuran: 96462
0x0e68432827674ad048b803d1 | CR sendirian: 0 | NUL: 0 | ukuran: 91433
0x116b1db5e116b7b9b5e2cd2c | CR sendirian: 0 | NUL: 0 | ukuran: 90235


In [ ]:
from datasets import load_dataset

target_addr = pola51.iloc[0].nama_file.replace('.sol', '').replace('0x', '')
files = [f"hf://datasets/mwritescode/slither-audited-smart-contracts/data/raw/contracts{i}.parquet" for i in range(9)]
ds = load_dataset("parquet", data_files=files, split="train", streaming=True)

for item in ds:
    if item.get("contracts", "").strip() == target_addr:
        sc = item["source_code"]
        print('panjang dari dataset :', len(sc))
        with open(pola51.iloc[0].path, encoding='utf-8', errors='replace') as f:
            lokal = f.read()
        print('panjang file lokal   :', len(lokal))
        print('identik              :', sc == lokal)
        print('\ncuplikan dataset di sekitar MINTER_ROLE:')
        i = sc.find('MINTER_ROLE')
        print(repr(sc[max(0,i-160):i+40]))
        break

panjang dari dataset : 44903
panjang file lokal   : 44903
identik              : True

cuplikan dataset di sekitar MINTER_ROLE:
'rtual override {\n\n        super._beforeTokenTransfer(from, to, amount);\n\n\n\n        require(!paused(), "ERC20Pausable: token transfer while paused");\n\n    }\n\n}\n\nMINTER_ROLE");\n\n    bytes32 public const'


In [ ]:
import os, shutil
import pandas as pd

BASE = '/content/data/raw/dataset_penelitian'
KARANTINA = '/content/data/raw/karantina'
os.makedirs(f'{KARANTINA}/0_normal', exist_ok=True)
os.makedirs(f'{KARANTINA}/1_reentrancy', exist_ok=True)

rusak = df[(df.status == 'GAGAL') & (df.split == 'train')]
pindah = 0
for _, r in rusak.iterrows():
    tujuan = os.path.join(KARANTINA, r.folder, r.nama_file)
    if os.path.exists(r.path):
        shutil.move(r.path, tujuan)
        pindah += 1
print(f'{pindah} kontrak bermasalah dipindah ke karantina')

for f in ['0_normal', '1_reentrancy']:
    n = len([x for x in os.listdir(f'{BASE}/{f}') if x.endswith('.sol')])
    print(f'{f}: {n} file tersisa')

86 kontrak bermasalah dipindah ke karantina
0_normal: 488 file tersisa
1_reentrancy: 176 file tersisa


In [ ]:
!pip install -q datasets

In [ ]:
import os, json
from datasets import load_dataset
from tqdm.auto import tqdm

LABEL_MAPPING = {
    "reentrancy-no-eth": "reentrancy", "reentrancy-eth": "reentrancy", "safe": "safe",
    "suicidal": "access-control", "arbitrary-send": "access-control",
    "unused-return": "unchecked-calls", "unchecked-lowlevel": "unchecked-calls",
    "msg-value-loop": "double-spending", "controlled-array-length": "access-control",
    "locked-ether": "locked-ether", "tx-origin": "access-control",
    "delegatecall-loop": "double-spending", "constant-function-state": "other",
    "divide-before-multiply": "arithmetic", "array-by-reference": "other",
    "incorrect-equality": "other", "rtlo": "other", "backdoor": "other",
    "weak-prng": "bad-randomness", "controlled-delegatecall": "access-control",
    "constant-function-asm": "other", "unprotected-upgrade": "access-control",
    "uninitialized-storage": "other", "unchecked-send": "locked-ether",
    "uninitialized-local": "other", "unchecked-transfer": "unchecked-calls",
    "mapping-deletion": "other", "uninitialized-state": "other",
}

def kontrak_utuh(sc):
    if not sc or len(sc) < 200:
        return False
    if sc.count('{') != sc.count('}'):
        return False
    if sc.count('(') != sc.count(')'):
        return False
    if sc.count('"') % 2 != 0:
        return False
    if 'pragma solidity' not in sc:
        return False
    return True

normal_dir = f'{BASE}/0_normal'
reent_dir = f'{BASE}/1_reentrancy'
kuota_normal, kuota_reent = 500, 250

n_normal = len([f for f in os.listdir(normal_dir) if f.endswith('.sol')])
n_reent = len([f for f in os.listdir(reent_dir) if f.endswith('.sol')])
print(f'Mulai dari {n_normal} normal, {n_reent} reentrancy')

# alamat yang sudah pernah dipakai atau dikarantina, jangan diambil lagi
sudah = set()
for d in [normal_dir, reent_dir, f'{KARANTINA}/0_normal', f'{KARANTINA}/1_reentrancy']:
    sudah |= {f.replace('.sol', '').lower() for f in os.listdir(d) if f.endswith('.sol')}
print(f'{len(sudah)} alamat dikecualikan')

files = [f"hf://datasets/mwritescode/slither-audited-smart-contracts/data/raw/contracts{i}.parquet" for i in range(9)]
ds = load_dataset("parquet", data_files=files, split="train", streaming=True)

ditolak = 0
pbar = tqdm(total=(kuota_normal - n_normal) + (kuota_reent - n_reent), desc='Kontrak baru')

for item in ds:
    if n_normal >= kuota_normal and n_reent >= kuota_reent:
        break
    addr_raw = item.get("contracts", "").strip()
    sc = item.get("source_code", "")
    if not addr_raw or not sc:
        continue
    address = f"0x{addr_raw}"
    if address.lower() in sudah:
        continue
    if not kontrak_utuh(sc):
        ditolak += 1
        continue

    try:
        detectors = (json.loads(item.get("results", "{}")).get("results") or {}).get("detectors", [])
    except Exception:
        detectors = []
    kelas = set()
    for d in detectors:
        m = LABEL_MAPPING.get(d.get("check"))
        if m:
            kelas.add(m)
    if not detectors:
        kelas.add("safe")

    if "reentrancy" in kelas and n_reent < kuota_reent:
        with open(f'{reent_dir}/{address}.sol', 'w', encoding='utf-8', newline='') as f:
            f.write(sc)
        n_reent += 1
        sudah.add(address.lower())
        pbar.update(1)
    elif "safe" in kelas and not kelas - {"safe"} and n_normal < kuota_normal:
        with open(f'{normal_dir}/{address}.sol', 'w', encoding='utf-8', newline='') as f:
            f.write(sc)
        n_normal += 1
        sudah.add(address.lower())
        pbar.update(1)

pbar.close()
print(f'\nSelesai: {n_normal} normal, {n_reent} reentrancy')
print(f'{ditolak} kontrak ditolak karena tidak utuh')

Mulai dari 488 normal, 176 reentrancy
750 alamat dikecualikan


Kontrak baru:   0%|          | 0/86 [00:00<?, ?it/s]


Selesai: 500 normal, 250 reentrancy
52 kontrak ditolak karena tidak utuh


In [ ]:
import os, csv, subprocess
from slither import Slither
import pandas as pd

os.chdir('/content')
df = pd.read_csv(MANIFEST2)
lama = set(df.nama_file)

baru = []
for folder, label in [('0_normal', 0), ('1_reentrancy', 1)]:
    d = f'{BASE}/{folder}'
    for fn in sorted(os.listdir(d)):
        if fn.endswith('.sol') and fn not in lama:
            fp = os.path.join(d, fn)
            pragma = get_pragma(fp)
            baru.append((resolve_solc_version_v2(pragma) or 'ZZZ', fp, fn, folder, label, 'train', pragma))
baru.sort()
print(f'{len(baru)} kontrak baru akan divalidasi')

rows, versi_aktif = [], None
for i, (ver, fp, fn, folder, label, split, pragma) in enumerate(baru, 1):
    status, alasan, n_fn, n_node = 'GAGAL', '', 0, 0
    if ver == 'ZZZ':
        alasan = 'pragma_tidak_ditemukan'
    else:
        try:
            if ver != versi_aktif:
                subprocess.run(['solc-select', 'use', ver], capture_output=True, check=True)
                versi_aktif = ver
            sl = Slither(fp)
            n_fn = sum(len(c.functions) for c in sl.contracts)
            n_node = sum(len(f.nodes) for c in sl.contracts for f in c.functions)
            status = 'OK' if n_fn > 0 else 'GAGAL'
            if n_fn == 0:
                alasan = 'tidak_ada_fungsi'
        except Exception as e:
            alasan = str(e).replace('\n', ' ')[:200]
    rows.append([fp, fn, folder, label, split, pragma, ver if ver != 'ZZZ' else '', status, alasan, n_fn, n_node])
    print(f'[{i}/{len(baru)}] {status} {ver} {fn[:28]} {alasan[:45]}')

86 kontrak baru akan divalidasi


[1/86] OK 0.4.10 0x01d9f4d104668cdc0b6d13c45d 


[2/86] OK 0.4.26 0x01E45B8D0c51f05F17385DD341 


[3/86] OK 0.4.26 0x01d9c3baeb4a7b22a8742e07e9 


[4/86] OK 0.4.26 0x01e0f57c42bf976676ea1daa1b 


[5/86] OK 0.4.26 0x0080d4b7be95b550a69e878934 


... (68 outputs left out)

[81/86] OK 0.8.7 0x00a8bE6Ea4888D6F232dFB84c3 
[82/86] GAGAL 0.8.8 0x0130D226eb32546f0BdfC9e900 Command '['solc-select', 'use', '0.8.8']' ret


[83/86] OK 0.8.9 0x01d73bc5fd31af6b01655bf29b 


[84/86] OK 0.8.9 0x00e07fd90e86855c3f9ebf7741 


[85/86] OK 0.8.9 0x014942f890d762F1A335649A54 
[86/86] OK 0.8.9 0x014d9a527fe5d11c178d702489 


In [ ]:
kolom = ['path','nama_file','folder','label','split','pragma','solc','status','alasan','jumlah_fungsi','jumlah_node']
df_baru = pd.DataFrame(rows, columns=kolom)

# buang baris lama yang kontraknya sudah dikarantina
df_bersih = df[~((df.status == 'GAGAL') & (df.split == 'train'))]
df_final = pd.concat([df_bersih, df_baru], ignore_index=True)

MANIFEST3 = '/content/drive/MyDrive/PROJECT/data/processed/manifest_final.csv'
df_final.to_csv(MANIFEST3, index=False)

print(pd.crosstab([df_final.split, df_final.folder], df_final.status))

status                               GAGAL   OK
split     folder                               
benchmark 3_expert_benchmark/aman       33    2
          3_expert_benchmark/rentan      2   29
train     0_normal                       0  500
          1_reentrancy                   6  244


In [ ]:
import shutil, os

gagal6 = df_final[(df_final.status == 'GAGAL') & (df_final.split == 'train')]
for _, r in gagal6.iterrows():
    tujuan = os.path.join(KARANTINA, r.folder, r.nama_file)
    if os.path.exists(r.path):
        shutil.move(r.path, tujuan)
print(f'{len(gagal6)} kontrak dipindah ke karantina')

for f in ['0_normal', '1_reentrancy']:
    n = len([x for x in os.listdir(f'{BASE}/{f}') if x.endswith('.sol')])
    print(f'{f}: {n} file')

0 kontrak dipindah ke karantina
0_normal: 500 file
1_reentrancy: 244 file


In [ ]:
os.chdir('/content')
lama = set(df_final.nama_file)

baru = []
for folder, label in [('0_normal', 0), ('1_reentrancy', 1)]:
    d = f'{BASE}/{folder}'
    for fn in sorted(os.listdir(d)):
        if fn.endswith('.sol') and fn not in lama:
            fp = os.path.join(d, fn)
            pragma = get_pragma(fp)
            baru.append((resolve_solc_version_v2(pragma) or 'ZZZ', fp, fn, folder, label, 'train', pragma))
baru.sort()
print(f'{len(baru)} kontrak baru divalidasi')

rows2, versi_aktif = [], None
for i, (ver, fp, fn, folder, label, split, pragma) in enumerate(baru, 1):
    status, alasan, n_fn, n_node = 'GAGAL', '', 0, 0
    if ver == 'ZZZ':
        alasan = 'pragma_tidak_ditemukan'
    else:
        try:
            if ver != versi_aktif:
                subprocess.run(['solc-select', 'use', ver], capture_output=True, check=True)
                versi_aktif = ver
            sl = Slither(fp)
            n_fn = sum(len(c.functions) for c in sl.contracts)
            n_node = sum(len(f.nodes) for c in sl.contracts for f in c.functions)
            status = 'OK' if n_fn > 0 else 'GAGAL'
            if n_fn == 0:
                alasan = 'tidak_ada_fungsi'
        except Exception as e:
            alasan = str(e).replace('\n', ' ')[:200]
    rows2.append([fp, fn, folder, label, split, pragma, ver if ver != 'ZZZ' else '', status, alasan, n_fn, n_node])
    print(f'[{i}/{len(baru)}] {status} {ver} {fn[:28]} {alasan[:45]}')

df_final = pd.concat([
    df_final[~((df_final.status == 'GAGAL') & (df_final.split == 'train'))],
    pd.DataFrame(rows2, columns=kolom)
], ignore_index=True)
df_final.to_csv(MANIFEST3, index=False)
print(pd.crosstab([df_final.split, df_final.folder], df_final.status))

0 kontrak baru divalidasi
status                               GAGAL   OK
split     folder                               
benchmark 3_expert_benchmark/aman       33    2
          3_expert_benchmark/rentan      2   29
train     0_normal                       0  500
          1_reentrancy                   0  244


In [ ]:
import os
d = f'{BASE}/3_expert_benchmark/aman'
files = sorted(os.listdir(d))
print(f'{len(files)} file\n')
for f in files[:20]:
    print(' ', f)

35 file

  AccessControlEnumerable.sol
  BridgeERC7802.sol
  BridgeMultiToken.sol
  Context.sol
  Create2.sol
  CrosschainLinked.sol
  CrosschainRemoteExecutor.sol
  EIP712.sol
  ERC1155Crosschain.sol
  ERC1155Pausable.sol
  ERC1155URIStorage.sol
  ERC1967Utils.sol
  ERC20FlashMint.sol
  ERC20Votes.sol
  ERC2771Context.sol
  ERC2771Forwarder.sol
  ERC2981.sol
  ERC721Holder.sol
  ERC7786Recipient.sol
  EnumerableSet.sol


In [ ]:
for f in files[:6]:
    print('=' * 60)
    print(f)
    with open(os.path.join(d, f), encoding='utf-8', errors='replace') as fh:
        for baris in fh.readlines()[:12]:
            print('  ', baris.rstrip()[:90])

AccessControlEnumerable.sol
   // SPDX-License-Identifier: MIT
   // OpenZeppelin Contracts (last updated v5.5.0) (access/extensions/AccessControlEnumerable
   
   pragma solidity ^0.8.24;
   
   import {IAccessControlEnumerable} from "./IAccessControlEnumerable.sol";
   import {AccessControl} from "../AccessControl.sol";
   import {EnumerableSet} from "../../utils/structs/EnumerableSet.sol";
   import {IERC165} from "../../utils/introspection/ERC165.sol";
   
   /**
    * @dev Extension of {AccessControl} that allows enumerating the members of each role.
BridgeERC7802.sol
   // SPDX-License-Identifier: MIT
   // OpenZeppelin Contracts (last updated v5.6.0) (crosschain/bridges/BridgeERC7802.sol)
   
   pragma solidity ^0.8.26;
   
   import {IERC7802} from "../../interfaces/draft-IERC7802.sol";
   import {BridgeFungible} from "./abstract/BridgeFungible.sol";
   
   /**
    * @dev This is a variant of {BridgeFungible} that implements the bridge logic for ERC-780
    */
   // slither-dis

In [ ]:
!git clone --depth 1 https://github.com/smartbugs/smartbugs-curated.git /content/smartbugs 2>&1 | tail -3
!find /content/smartbugs -type d | head -20
!find /content/smartbugs -name "*.sol" | wc -l

Cloning into '/content/smartbugs'...
/content/smartbugs
/content/smartbugs/.git
/content/smartbugs/.git/branches
/content/smartbugs/.git/logs
/content/smartbugs/.git/logs/refs
/content/smartbugs/.git/logs/refs/remotes
/content/smartbugs/.git/logs/refs/remotes/origin
/content/smartbugs/.git/logs/refs/heads
/content/smartbugs/.git/refs
/content/smartbugs/.git/refs/remotes
/content/smartbugs/.git/refs/remotes/origin
/content/smartbugs/.git/refs/tags
/content/smartbugs/.git/refs/heads
/content/smartbugs/.git/hooks
/content/smartbugs/.git/info
/content/smartbugs/.git/objects
/content/smartbugs/.git/objects/pack
/content/smartbugs/.git/objects/info
/content/smartbugs/dataset
/content/smartbugs/dataset/denial_of_service
143


In [ ]:
import os
d = '/content/smartbugs/dataset'
print(sorted(os.listdir(d)))
print()
re_dir = f'{d}/reentrancy'
files_sb = sorted(os.listdir(re_dir))
print(f'{len(files_sb)} file di folder reentrancy:')
for f in files_sb:
    print(' ', f)

['access_control', 'arithmetic', 'bad_randomness', 'denial_of_service', 'front_running', 'other', 'reentrancy', 'short_addresses', 'time_manipulation', 'unchecked_low_level_calls']

32 file di folder reentrancy:
  0x01f8c4e3fa3edeb29e514cba738d87ce8c091d3f.sol
  0x23a91059fdc9579a9fbd0edc5f2ea0bfdb70deb4.sol
  0x4320e6f8c05b27ab4707cd1f6d5ce6f3e4b3a5a1.sol
  0x4e73b32ed6c35f570686b89848e5f39f20ecc106.sol
  0x561eac93c92360949ab1f1403323e6db345cbf31.sol
  0x627fa62ccbb1c1b04ffaecd72a53e37fc0e17839.sol
  0x7541b76cb60f4c60af330c208b0623b7f54bf615.sol
  0x7a8721a9d64c74da899424c1b52acbf58ddc9782.sol
  0x7b368c4e805c3870b6c49a3f1f49f69af8662cf3.sol
  0x8c7777c45481dba411450c228cb692ac3d550344.sol
  0x93c32845fae42c83a70e5f06214c8433665c2ab5.sol
  0x941d225236464a25eb18076df7da6a91d0f95e9e.sol
  0x96edbe868531bd23a6c05e9d0c424ea64fb1b78b.sol
  0xaae1f51cf3339f18b6d3f3bdc75a5facd744b0b8.sol
  0xb5e1b1ee15c6fa0e48fce100125569d430f1bd12.sol
  0xb93430ce38ac4a6bb47fb1fc085ea669353fd89e.sol
  0x

In [ ]:
punya = {f.lower() for f in os.listdir(f'{BASE}/3_expert_benchmark/rentan') if f.endswith('.sol')}
sb = {f.lower() for f in files_sb}
print(f'punya kamu: {len(punya)}, di SmartBugs: {len(sb)}')
print(f'beririsan : {len(punya & sb)}')
print(f'hanya di SmartBugs: {sorted(sb - punya)}')

punya kamu: 31, di SmartBugs: 32
beririsan : 31
hanya di SmartBugs: ['readme.md']


In [ ]:
for f in ['simple_dao.sol', 'reentrance.sol', 'reentrancy_simple.sol', 'etherstore.sol']:
    p = f'/content/smartbugs/dataset/reentrancy/{f}'
    print('=' * 70)
    print(f)
    with open(p, encoding='utf-8', errors='replace') as fh:
        print(fh.read())

simple_dao.sol
/*
 * @source: http://blockchain.unica.it/projects/ethereum-survey/attacks.html#simpledao
 * @author: -
 * @vulnerable_at_lines: 19
 */

pragma solidity ^0.4.2;

contract SimpleDAO {
  mapping (address => uint) public credit;

  function donate(address to) payable {
    credit[to] += msg.value;
  }

  function withdraw(uint amount) {
    if (credit[msg.sender]>= amount) {
      // <yes> <report> REENTRANCY
      bool res = msg.sender.call.value(amount)();
      credit[msg.sender]-=amount;
    }
  }

  function queryCredit(address to) returns (uint){
    return credit[to];
  }
}

reentrance.sol
/*
 * @source: https://ethernaut.zeppelin.solutions/level/0xf70706db003e94cfe4b5e27ffd891d5c81b39488
 * @author: Alejandro Santander
 * @vulnerable_at_lines: 24
 */

pragma solidity ^0.4.18;

contract Reentrance {

  mapping(address => uint) public balances;

  function donate(address _to) public payable {
    balances[_to] += msg.value;
  }

  function balanceOf(address _who) publ

In [ ]:
import os
AMAN = f'{BASE}/3_expert_benchmark/aman'
os.makedirs(AMAN, exist_ok=True)

fixed = {}

fixed['simple_dao_fixed.sol'] = '''pragma solidity ^0.4.2;

contract SimpleDAO {
  mapping (address => uint) public credit;

  function donate(address to) payable {
    credit[to] += msg.value;
  }

  function withdraw(uint amount) {
    if (credit[msg.sender]>= amount) {
      credit[msg.sender]-=amount;
      bool res = msg.sender.call.value(amount)();
    }
  }

  function queryCredit(address to) returns (uint){
    return credit[to];
  }
}
'''

fixed['reentrance_fixed.sol'] = '''pragma solidity ^0.4.18;

contract Reentrance {

  mapping(address => uint) public balances;

  function donate(address _to) public payable {
    balances[_to] += msg.value;
  }

  function balanceOf(address _who) public view returns (uint balance) {
    return balances[_who];
  }

  function withdraw(uint _amount) public {
    if(balances[msg.sender] >= _amount) {
      balances[msg.sender] -= _amount;
      if(msg.sender.call.value(_amount)()) {
        _amount;
      }
    }
  }

  function() public payable {}
}
'''

fixed['reentrancy_simple_fixed.sol'] = '''pragma solidity ^0.4.15;

contract Reentrance {
    mapping (address => uint) userBalance;

    function getBalance(address u) constant returns(uint){
        return userBalance[u];
    }

    function addToBalance() payable{
        userBalance[msg.sender] += msg.value;
    }

    function withdrawBalance(){
        uint amount = userBalance[msg.sender];
        userBalance[msg.sender] = 0;
        if( ! (msg.sender.call.value(amount)() ) ){
            throw;
        }
    }
}
'''

fixed['etherstore_fixed.sol'] = '''pragma solidity ^0.4.10;

contract EtherStore {

    uint256 public withdrawalLimit = 1 ether;
    mapping(address => uint256) public lastWithdrawTime;
    mapping(address => uint256) public balances;

    function depositFunds() public payable {
        balances[msg.sender] += msg.value;
    }

    function withdrawFunds (uint256 _weiToWithdraw) public {
        require(balances[msg.sender] >= _weiToWithdraw);
        require(_weiToWithdraw <= withdrawalLimit);
        require(now >= lastWithdrawTime[msg.sender] + 1 weeks);
        balances[msg.sender] -= _weiToWithdraw;
        lastWithdrawTime[msg.sender] = now;
        require(msg.sender.call.value(_weiToWithdraw)());
    }
 }
'''

for nama, isi in fixed.items():
    with open(os.path.join(AMAN, nama), 'w', encoding='utf-8', newline='') as f:
        f.write(isi)
print(f'{len(fixed)} kontrak versi CEI ditulis')

4 kontrak versi CEI ditulis


In [ ]:
import subprocess, json

for nama in fixed:
    fp = os.path.join(AMAN, nama)
    ver = resolve_solc_version_v2(get_pragma(fp))
    subprocess.run(['solc-select', 'use', ver], capture_output=True)
    out = subprocess.run(
        ['slither', fp, '--detect', 'reentrancy-eth,reentrancy-no-eth', '--json', '-'],
        capture_output=True, text=True
    )
    try:
        hasil = json.loads(out.stdout)
        temuan = hasil.get('results', {}).get('detectors', [])
    except Exception:
        temuan = ['<gagal parse>']
    print(f'{nama:32} solc {ver:8} temuan reentrancy: {len(temuan)}')

simple_dao_fixed.sol             solc 0.4.26   temuan reentrancy: 0
reentrance_fixed.sol             solc 0.4.26   temuan reentrancy: 0
reentrancy_simple_fixed.sol      solc 0.4.26   temuan reentrancy: 0
etherstore_fixed.sol             solc 0.4.26   temuan reentrancy: 0


In [ ]:
import os, subprocess, json

AMAN = f'{BASE}/3_expert_benchmark/aman'
sudah_jadi = ['simple_dao_fixed.sol', 'reentrance_fixed.sol',
              'reentrancy_simple_fixed.sol', 'etherstore_fixed.sol']

for nama in sudah_jadi:
    fp = os.path.join(AMAN, nama)
    if not os.path.exists(fp):
        print(f'{nama:34} TIDAK ADA')
        continue
    ver = resolve_solc_version_v2(get_pragma(fp))
    subprocess.run(['solc-select', 'use', ver], capture_output=True)
    out = subprocess.run(
        ['slither', fp, '--detect', 'reentrancy-eth,reentrancy-no-eth', '--json', '-'],
        capture_output=True, text=True
    )
    try:
        temuan = json.loads(out.stdout).get('results', {}).get('detectors', [])
        status = 'AMAN' if len(temuan) == 0 else f'MASIH RENTAN ({len(temuan)})'
    except Exception:
        status = 'gagal parse, cek stderr'
    print(f'{nama:34} solc {ver:8} {status}')

simple_dao_fixed.sol               solc 0.4.26   AMAN
reentrance_fixed.sol               solc 0.4.26   AMAN
reentrancy_simple_fixed.sol        solc 0.4.26   AMAN
etherstore_fixed.sol               solc 0.4.26   AMAN


In [ ]:
RENTAN = '/content/smartbugs/dataset/reentrancy'
for nama in ['simple_dao.sol', 'reentrance.sol', 'reentrancy_simple.sol', 'etherstore.sol']:
    fp = os.path.join(RENTAN, nama)
    ver = resolve_solc_version_v2(get_pragma(fp))
    subprocess.run(['solc-select', 'use', ver], capture_output=True)
    out = subprocess.run(
        ['slither', fp, '--detect', 'reentrancy-eth,reentrancy-no-eth', '--json', '-'],
        capture_output=True, text=True
    )
    try:
        temuan = json.loads(out.stdout).get('results', {}).get('detectors', [])
    except Exception:
        temuan = []
    print(f'{nama:30} temuan: {len(temuan)}')

simple_dao.sol                 temuan: 1
reentrance.sol                 temuan: 1
reentrancy_simple.sol          temuan: 1
etherstore.sol                 temuan: 1


In [ ]:
sisa = ['etherbank.sol', 'reentrancy_insecure.sol', 'reentrancy_dao.sol',
        'reentrancy_bonus.sol', 'reentrancy_cross_function.sol',
        'modifier_reentrancy.sol', 'spank_chain_payment.sol']

for f in sisa[:4]:
    p = os.path.join(RENTAN, f)
    print('=' * 70)
    print(f)
    with open(p, encoding='utf-8', errors='replace') as fh:
        print(fh.read())

etherbank.sol
/*
 * @source: https://github.com/seresistvanandras/EthBench/blob/master/Benchmark/Simple/reentrant.sol
 * @author: -
 * @vulnerable_at_lines: 21
 */

pragma solidity ^0.4.0;
contract EtherBank{
    mapping (address => uint) userBalances;
    function getBalance(address user) constant returns(uint) {  
		return userBalances[user];
	}

	function addToBalance() {  
		userBalances[msg.sender] += msg.value;
	}

	function withdrawBalance() {  
		uint amountToWithdraw = userBalances[msg.sender];
        // <yes> <report> REENTRANCY
		if (!(msg.sender.call.value(amountToWithdraw)())) { throw; }
		userBalances[msg.sender] = 0;
	}    
}
reentrancy_insecure.sol
/*
 * @source: https://consensys.github.io/smart-contract-best-practices/known_attacks/
 * @author: consensys
 * @vulnerable_at_lines: 17
 */

pragma solidity ^0.5.0;

contract Reentrancy_insecure {

    // INSECURE
    mapping (address => uint) private userBalances;

    function withdrawBalance() public {
        uint amou

In [ ]:
fixed2 = {}

fixed2['etherbank_fixed.sol'] = '''pragma solidity ^0.4.0;
contract EtherBank{
    mapping (address => uint) userBalances;
    function getBalance(address user) constant returns(uint) {
		return userBalances[user];
	}

	function addToBalance() {
		userBalances[msg.sender] += msg.value;
	}

	function withdrawBalance() {
		uint amountToWithdraw = userBalances[msg.sender];
		userBalances[msg.sender] = 0;
		if (!(msg.sender.call.value(amountToWithdraw)())) { throw; }
	}
}
'''

fixed2['reentrancy_insecure_fixed.sol'] = '''pragma solidity ^0.5.0;

contract Reentrancy_insecure {

    mapping (address => uint) private userBalances;

    function withdrawBalance() public {
        uint amountToWithdraw = userBalances[msg.sender];
        userBalances[msg.sender] = 0;
        (bool success, ) = msg.sender.call.value(amountToWithdraw)("");
        require(success);
    }
}
'''

fixed2['reentrancy_dao_fixed.sol'] = '''pragma solidity ^0.4.19;

contract ReentrancyDAO {
    mapping (address => uint) credit;
    uint balance;

    function withdrawAll() public {
        uint oCredit = credit[msg.sender];
        if (oCredit > 0) {
            balance -= oCredit;
            credit[msg.sender] = 0;
            bool callResult = msg.sender.call.value(oCredit)();
            require (callResult);
        }
    }

    function deposit() public payable {
        credit[msg.sender] += msg.value;
        balance += msg.value;
    }
}
'''

fixed2['reentrancy_bonus_fixed.sol'] = '''pragma solidity ^0.4.24;

contract Reentrancy_bonus{

    mapping (address => uint) private userBalances;
    mapping (address => bool) private claimedBonus;
    mapping (address => uint) private rewardsForA;

    function withdrawReward(address recipient) public {
        uint amountToWithdraw = rewardsForA[recipient];
        rewardsForA[recipient] = 0;
        (bool success, ) = recipient.call.value(amountToWithdraw)("");
        require(success);
    }

    function getFirstWithdrawalBonus(address recipient) public {
        require(!claimedBonus[recipient]);

        rewardsForA[recipient] += 100;
        claimedBonus[recipient] = true;
        withdrawReward(recipient);
    }
}
'''

for nama, isi in fixed2.items():
    with open(os.path.join(AMAN, nama), 'w', encoding='utf-8', newline='') as f:
        f.write(isi)
print(f'{len(fixed2)} kontrak versi CEI ditulis')

4 kontrak versi CEI ditulis


In [ ]:
import subprocess, json

def cek_reentrancy(fp):
    ver = resolve_solc_version_v2(get_pragma(fp))
    subprocess.run(['solc-select', 'use', ver], capture_output=True)
    out = subprocess.run(
        ['slither', fp, '--detect', 'reentrancy-eth,reentrancy-no-eth', '--json', '-'],
        capture_output=True, text=True
    )
    try:
        return ver, len(json.loads(out.stdout).get('results', {}).get('detectors', []))
    except Exception:
        return ver, -1

pasangan = [
    ('simple_dao.sol', 'simple_dao_fixed.sol'),
    ('reentrance.sol', 'reentrance_fixed.sol'),
    ('reentrancy_simple.sol', 'reentrancy_simple_fixed.sol'),
    ('etherstore.sol', 'etherstore_fixed.sol'),
    ('etherbank.sol', 'etherbank_fixed.sol'),
    ('reentrancy_insecure.sol', 'reentrancy_insecure_fixed.sol'),
    ('reentrancy_dao.sol', 'reentrancy_dao_fixed.sol'),
    ('reentrancy_bonus.sol', 'reentrancy_bonus_fixed.sol'),
]

for asal, fix in pasangan:
    v1, n1 = cek_reentrancy(os.path.join(RENTAN, asal))
    v2, n2 = cek_reentrancy(os.path.join(AMAN, fix))
    tanda = 'OK' if (n1 > 0 and n2 == 0) else 'PERIKSA'
    print(f'{asal:30} rentan:{n1:3}  ->  fixed:{n2:3}  solc {v2:8} {tanda}')

simple_dao.sol                 rentan:  1  ->  fixed:  0  solc 0.4.26   OK
reentrance.sol                 rentan:  1  ->  fixed:  0  solc 0.4.26   OK
reentrancy_simple.sol          rentan:  1  ->  fixed:  0  solc 0.4.26   OK
etherstore.sol                 rentan:  1  ->  fixed:  0  solc 0.4.26   OK
etherbank.sol                  rentan:  1  ->  fixed:  0  solc 0.4.26   OK
reentrancy_insecure.sol        rentan:  1  ->  fixed:  0  solc 0.5.17   OK
reentrancy_dao.sol             rentan:  1  ->  fixed:  0  solc 0.4.26   OK
reentrancy_bonus.sol           rentan: -1  ->  fixed: -1  solc 0.4.26   PERIKSA


In [ ]:
p = os.path.join(AMAN, 'reentrancy_bonus_fixed.sol')
if os.path.exists(p):
    os.remove(p)
    print('reentrancy_bonus_fixed.sol dihapus')

print('\nIsi folder aman sekarang:')
for f in sorted(os.listdir(AMAN)):
    print(' ', f)

reentrancy_bonus_fixed.sol dihapus

Isi folder aman sekarang:
  AccessControlEnumerable.sol
  BridgeERC7802.sol
  BridgeMultiToken.sol
  Context.sol
  Create2.sol
  CrosschainLinked.sol
  CrosschainRemoteExecutor.sol
  EIP712.sol
  ERC1155Crosschain.sol
  ERC1155Pausable.sol
  ERC1155URIStorage.sol
  ERC1967Utils.sol
  ERC20FlashMint.sol
  ERC20Votes.sol
  ERC2771Context.sol
  ERC2771Forwarder.sol
  ERC2981.sol
  ERC721Holder.sol
  ERC7786Recipient.sol
  EnumerableSet.sol
  GovernorProposalGuardian.sol
  GovernorSettings.sol
  GovernorVotes.sol
  GovernorVotesQuorumFraction.sol
  GovernorVotesSuperQuorumFraction.sol
  PaymasterERC721Owner.sol
  PaymasterSigner.sol
  ShortStrings.sol
  SlotDerivation.sol
  TimelockController.sol
  TransparentUpgradeableProxy.sol
  VestingWallet.sol
  WebAuthn.sol
  draft-AccountERC7579.sol
  draft-AccountERC7579Hooked.sol
  etherbank_fixed.sol
  etherstore_fixed.sol
  reentrance_fixed.sol
  reentrancy_dao_fixed.sol
  reentrancy_insecure_fixed.sol
  reen

In [ ]:
import shutil
os.makedirs(f'{KARANTINA}/3_expert_benchmark_aman_oz', exist_ok=True)
pindah = 0
for f in sorted(os.listdir(AMAN)):
    if f.endswith('.sol') and not f.endswith('_fixed.sol'):
        shutil.move(os.path.join(AMAN, f), f'{KARANTINA}/3_expert_benchmark_aman_oz/{f}')
        pindah += 1
print(f'{pindah} file OpenZeppelin dipindah ke karantina')
print(f'tersisa {len([f for f in os.listdir(AMAN) if f.endswith(".sol")])} kontrak aman')

# kontrak rentan yang pasangannya tidak jadi, keluarkan juga dari benchmark
rentan_dir = f'{BASE}/3_expert_benchmark/rentan'
p2 = os.path.join(rentan_dir, 'reentrancy_bonus.sol')
if os.path.exists(p2):
    shutil.move(p2, f'{KARANTINA}/3_expert_benchmark_aman_oz/reentrancy_bonus.sol')
    print('reentrancy_bonus.sol dikeluarkan dari benchmark rentan')

35 file OpenZeppelin dipindah ke karantina
tersisa 7 kontrak aman
reentrancy_bonus.sol dikeluarkan dari benchmark rentan


In [ ]:
for f in ['reentrancy_cross_function.sol', 'modifier_reentrancy.sol']:
    p = os.path.join(RENTAN, f)
    print('=' * 70)
    print(f)
    with open(p, encoding='utf-8', errors='replace') as fh:
        print(fh.read())

reentrancy_cross_function.sol
/*
 * @source: https://consensys.github.io/smart-contract-best-practices/known_attacks/
 * @author: consensys
 * @vulnerable_at_lines: 24
 */

pragma solidity ^0.4.24;

contract Reentrancy_cross_function {

    // INSECURE
    mapping (address => uint) private userBalances;

    function transfer(address to, uint amount) {
        if (userBalances[msg.sender] >= amount) {
            userBalances[to] += amount;
            userBalances[msg.sender] -= amount;
        }
    }

    function withdrawBalance() public {
        uint amountToWithdraw = userBalances[msg.sender];
        // <yes> <report> REENTRANCY
        (bool success, ) = msg.sender.call.value(amountToWithdraw)(""); // At this point, the caller's code is executed, and can call transfer()
        require(success);
        userBalances[msg.sender] = 0;
    }
}

modifier_reentrancy.sol
/*
 * @source: https://github.com/SmartContractSecurity/SWC-registry/blob/master/test_cases/reentracy/modifier_re

In [ ]:
v, n = cek_reentrancy(os.path.join(RENTAN, 'reentrancy_cross_function.sol'))
print(f'solc {v}, temuan: {n}')

solc 0.4.26, temuan: -1


In [ ]:
fixed3 = {}

fixed3['modifier_reentrancy_fixed.sol'] = '''pragma solidity ^0.4.24;

contract ModifierEntrancy {
  mapping (address => uint) public tokenBalance;
  string constant name = "Nu Token";

  function airDrop() hasNoBalance public{
    tokenBalance[msg.sender] += 20;
    require(keccak256(abi.encodePacked("Nu Token")) == Bank(msg.sender).supportsToken());
  }

  modifier hasNoBalance {
      require(tokenBalance[msg.sender] == 0);
      _;
  }
}

contract Bank{
    function supportsToken() external pure returns(bytes32){
        return(keccak256(abi.encodePacked("Nu Token")));
    }
}
'''

for nama, isi in fixed3.items():
    with open(os.path.join(AMAN, nama), 'w', encoding='utf-8', newline='') as f:
        f.write(isi)

v1, n1 = cek_reentrancy(os.path.join(RENTAN, 'modifier_reentrancy.sol'))
v2, n2 = cek_reentrancy(os.path.join(AMAN, 'modifier_reentrancy_fixed.sol'))
print(f'modifier_reentrancy  rentan:{n1} -> fixed:{n2}  solc {v2}')

modifier_reentrancy  rentan:1 -> fixed:0  solc 0.4.26


In [ ]:
v, n = cek_reentrancy(os.path.join(RENTAN, 'reentrancy_cross_function.sol'))
print(f'solc {v}, temuan: {n}')

solc 0.4.26, temuan: -1


In [ ]:
fixed3 = {}

fixed3['modifier_reentrancy_fixed.sol'] = '''pragma solidity ^0.4.24;

contract ModifierEntrancy {
  mapping (address => uint) public tokenBalance;
  string constant name = "Nu Token";

  function airDrop() hasNoBalance public{
    tokenBalance[msg.sender] += 20;
    require(keccak256(abi.encodePacked("Nu Token")) == Bank(msg.sender).supportsToken());
  }

  modifier hasNoBalance {
      require(tokenBalance[msg.sender] == 0);
      _;
  }
}

contract Bank{
    function supportsToken() external pure returns(bytes32){
        return(keccak256(abi.encodePacked("Nu Token")));
    }
}
'''

for nama, isi in fixed3.items():
    with open(os.path.join(AMAN, nama), 'w', encoding='utf-8', newline='') as f:
        f.write(isi)

v1, n1 = cek_reentrancy(os.path.join(RENTAN, 'modifier_reentrancy.sol'))
v2, n2 = cek_reentrancy(os.path.join(AMAN, 'modifier_reentrancy_fixed.sol'))
print(f'modifier_reentrancy  rentan:{n1} -> fixed:{n2}  solc {v2}')

modifier_reentrancy  rentan:1 -> fixed:0  solc 0.4.26


In [ ]:
p = os.path.join(RENTAN, 'spank_chain_payment.sol')
with open(p, encoding='utf-8', errors='replace') as fh:
    isi = fh.read()
print(f'{len(isi)} karakter, {isi.count(chr(10))} baris')

if isi.count('\n') > 300:
    baris = isi.split('\n')
    for i, b in enumerate(baris):
        if '<yes> <report>' in b:
            awal, akhir = max(0, i - 25), min(len(baris), i + 15)
            print(f'\n--- sekitar baris {i+1} ---')
            print('\n'.join(baris[awal:akhir]))
else:
    print('\n' + isi)

36387 karakter, 896 baris

--- sekitar baris 425 ---
         if(_balances[0] != 0) {
             require(msg.value == _balances[0], "Eth balance does not match sent value");
             Channels[_lcID].ethBalances[0] = msg.value;
         }
         if(_balances[1] != 0) {
             Channels[_lcID].token = HumanStandardToken(_token);
             require(Channels[_lcID].token.transferFrom(msg.sender, this, _balances[1]),"CreateChannel: token transfer failure");
             Channels[_lcID].erc20Balances[0] = _balances[1];
         }

         Channels[_lcID].sequence = 0;
         Channels[_lcID].confirmTime = _confirmTime;
         // is close flag, lc state sequence, number open vc, vc root hash, partyA...
         //Channels[_lcID].stateHash = keccak256(uint256(0), uint256(0), uint256(0), bytes32(0x0), bytes32(msg.sender), bytes32(_partyI), balanceA, balanceI);
         Channels[_lcID].LCopenTimeout = now + _confirmTime;
         Channels[_lcID].initialDeposit = _balances;

  

In [ ]:
import os, shutil, subprocess
from slither import Slither
import pandas as pd

# keluarkan kontrak rentan yang tidak punya pasangan
tanpa_pasangan = ['reentrancy_bonus.sol', 'reentrancy_cross_function.sol', 'spank_chain_payment.sol']
rentan_dir = f'{BASE}/3_expert_benchmark/rentan'
for f in tanpa_pasangan:
    p = os.path.join(rentan_dir, f)
    if os.path.exists(p):
        shutil.move(p, f'{KARANTINA}/3_expert_benchmark_aman_oz/{f}')
        print(f'{f} dikeluarkan')

# samakan isi folder rentan dengan 8 kontrak yang berpasangan
pasangan_valid = ['simple_dao', 'reentrance', 'reentrancy_simple', 'etherstore',
                  'etherbank', 'reentrancy_insecure', 'reentrancy_dao', 'modifier_reentrancy']

for f in sorted(os.listdir(rentan_dir)):
    if f.endswith('.sol') and f.replace('.sol', '') not in pasangan_valid:
        shutil.move(os.path.join(rentan_dir, f), f'{KARANTINA}/3_expert_benchmark_aman_oz/{f}')

print(f'\nrentan: {len([f for f in os.listdir(rentan_dir) if f.endswith(".sol")])}')
print(f'aman  : {len([f for f in os.listdir(AMAN) if f.endswith(".sol")])}')

reentrancy_cross_function.sol dikeluarkan
spank_chain_payment.sol dikeluarkan

rentan: 8
aman  : 8


In [ ]:
os.chdir('/content')
rows_bm, versi_aktif = [], None
for folder, label in [('3_expert_benchmark/aman', 0), ('3_expert_benchmark/rentan', 1)]:
    d = f'{BASE}/{folder}'
    for fn in sorted(os.listdir(d)):
        if not fn.endswith('.sol'):
            continue
        fp = os.path.join(d, fn)
        pragma = get_pragma(fp)
        ver = resolve_solc_version_v2(pragma)
        status, alasan, n_fn, n_node = 'GAGAL', '', 0, 0
        try:
            if ver != versi_aktif:
                subprocess.run(['solc-select', 'use', ver], capture_output=True, check=True)
                versi_aktif = ver
            sl = Slither(fp)
            n_fn = sum(len(c.functions) for c in sl.contracts)
            n_node = sum(len(f.nodes) for c in sl.contracts for f in c.functions)
            status = 'OK' if n_fn > 0 else 'GAGAL'
        except Exception as e:
            alasan = str(e).replace('\n', ' ')[:200]
        rows_bm.append([fp, fn, folder, label, 'benchmark', pragma, ver, status, alasan, n_fn, n_node])
        print(f'{status} {fn[:34]} {alasan[:40]}')

df_final = pd.concat([
    df_final[df_final.split != 'benchmark'],
    pd.DataFrame(rows_bm, columns=kolom)
], ignore_index=True)
df_final.to_csv(MANIFEST3, index=False)
print()
print(pd.crosstab([df_final.split, df_final.folder], df_final.status))

OK etherbank_fixed.sol 


OK etherstore_fixed.sol 


OK modifier_reentrancy_fixed.sol 


OK reentrance_fixed.sol 
OK reentrancy_dao_fixed.sol 


OK reentrancy_insecure_fixed.sol 


OK reentrancy_simple_fixed.sol 


... (3 outputs left out)

OK modifier_reentrancy.sol 


OK reentrance.sol 
OK reentrancy_dao.sol 


OK reentrancy_insecure.sol 


OK reentrancy_simple.sol 
OK simple_dao.sol 

status                                OK
split     folder                        
benchmark 3_expert_benchmark/aman      8
          3_expert_benchmark/rentan    8
train     0_normal                   500
          1_reentrancy               244


In [ ]:
!cd /content/data/raw && zip -qr /content/drive/MyDrive/PROJECT/data/raw/dataset_final.zip dataset_penelitian
!ls -lh /content/drive/MyDrive/PROJECT/data/raw/

total 14M
-rw------- 1 root root 6.5M Sep 23 07:27 dataset_final.zip
-rw------- 1 root root 7.3M Aug 26 06:37 dataset_penelitian.zip


In [ ]:
!unzip -l /content/drive/MyDrive/PROJECT/data/raw/dataset_final.zip | tail -3

    13012  2026-09-23 06:10   dataset_penelitian/1_reentrancy/0x007851a6a74f016bfca89ea569072a42a6defa77.sol
---------                     -------
 31218360                     1267 files


In [ ]:
import os
print('BASE ada   :', os.path.exists('/content/data/raw/dataset_penelitian'))
print('karantina  :', os.path.exists('/content/data/raw/karantina'))
print('smartbugs  :', os.path.exists('/content/smartbugs'))

BASE ada   : True
karantina  : True
smartbugs  : True


In [ ]:
import os
for root, dirs, files in os.walk(BASE):
    sol = [f for f in files if f.endswith('.sol')]
    if sol:
        print(f'{root.replace(BASE, ""):40} {len(sol)}')

/0_normal                                500
/2_wild_test                             500
/3_expert_benchmark/rentan               8
/3_expert_benchmark/aman                 8
/1_reentrancy                            244


In [ ]:
import shutil, os
KARANTINA = '/content/data/raw/karantina'
wild = f'{BASE}/2_wild_test'
if os.path.exists(wild):
    shutil.move(wild, f'{KARANTINA}/2_wild_test')
    print('2_wild_test dipindah ke karantina')

!rm -f /content/drive/MyDrive/PROJECT/data/raw/dataset_final.zip
!cd /content/data/raw && zip -qr /content/drive/MyDrive/PROJECT/data/raw/dataset_final.zip dataset_penelitian
!cd /content/data/raw && zip -qr /content/drive/MyDrive/PROJECT/data/raw/karantina.zip karantina
!unzip -l /content/drive/MyDrive/PROJECT/data/raw/dataset_final.zip | tail -2
!ls -lh /content/drive/MyDrive/PROJECT/data/raw/

2_wild_test dipindah ke karantina
---------                     -------
 24199535                     766 files
total 16M
-rw------- 1 root root 4.8M Sep 23 07:31 dataset_final.zip
-rw------- 1 root root 7.3M Aug 26 06:37 dataset_penelitian.zip
-rw------- 1 root root 3.2M Sep 23 07:31 karantina.zip


In [ ]:
import os
D = '/content/drive/MyDrive/PROJECT'

cek = [
    f'{D}/data/raw/dataset_final.zip',
    f'{D}/data/raw/karantina.zip',
    f'{D}/data/processed/manifest_final.csv',
    f'{D}/solc_binaries',
]
for p in cek:
    ada = os.path.exists(p)
    ukuran = ''
    if ada and os.path.isfile(p):
        ukuran = f'{os.path.getsize(p)/1024/1024:.1f} MB'
    elif ada:
        ukuran = f'{len(os.listdir(p))} item'
    print(f'{"OK " if ada else "TIDAK ADA"} {p.replace(D, "")}  {ukuran}')

OK  /data/raw/dataset_final.zip  4.7 MB
OK  /data/raw/karantina.zip  3.1 MB
OK  /data/processed/manifest_final.csv  0.1 MB
OK  /solc_binaries  0 item


In [ ]:
import pandas as pd
df_cek = pd.read_csv(f'{D}/data/processed/manifest_final.csv')
print(f'{len(df_cek)} baris')
print(pd.crosstab([df_cek.split, df_cek.folder], df_cek.status))

760 baris
status                                OK
split     folder                        
benchmark 3_expert_benchmark/aman      8
          3_expert_benchmark/rentan    8
train     0_normal                   500
          1_reentrancy               244



SET UP COLLAB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q slither-analyzer solc-select torch_geometric

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.0/765.0 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.3/586.3 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os, pathlib
D = '/content/drive/MyDrive/PROJECT'
link = pathlib.Path.home() / '.solc-select'
if not link.exists():
    os.symlink(f'{D}/solc_binaries', link)
!solc-select versions | head -5

No solc version installed. Run `solc-select install --help` for more information


In [ ]:
import os
p = '/content/drive/MyDrive/PROJECT/solc_binaries'
print('isi solc_binaries:', os.listdir(p)[:10])
print('jumlah item      :', len(os.listdir(p)))

for sub in os.listdir(p)[:3]:
    q = os.path.join(p, sub)
    if os.path.isdir(q):
        print(f'  {sub}/ ->', os.listdir(q)[:5])

isi solc_binaries: ['artifacts']
jumlah item      : 1
  artifacts/ -> []


In [ ]:
import os
a = '/content/drive/MyDrive/PROJECT/solc_binaries/artifacts'
isi = sorted(os.listdir(a))
print(f'{len(isi)} item:', isi[:5])

if isi:
    sub = os.path.join(a, isi[0])
    print(f'\n{isi[0]} ->', os.listdir(sub) if os.path.isdir(sub) else 'bukan folder')

!ls -la ~/.solc-select/
!ls -la ~/.solc-select/artifacts/ | head -5

0 item: []
total 12
drwx------ 3 root root 4096 Sep 23 09:48 .
drwx------ 4 root root 4096 Aug 26 05:34 ..
drwx------ 2 root root 4096 Sep 23 09:48 artifacts
total 8
drwx------ 2 root root 4096 Sep 23 09:48 .
drwx------ 3 root root 4096 Sep 23 09:48 ..


In [ ]:
import os, pathlib, shutil
link = pathlib.Path.home() / '.solc-select'
if link.is_symlink():
    link.unlink()
    print('symlink dilepas')
os.makedirs(link / 'artifacts', exist_ok=True)
!solc-select versions | head -3

symlink dilepas
No solc version installed. Run `solc-select install --help` for more information


In [ ]:
import pandas as pd
df_final = pd.read_csv(f'{D}/data/processed/manifest_final.csv')
versi = sorted(df_final.solc.dropna().unique())
print(f'{len(versi)} versi dibutuhkan:', versi)

61 versi dibutuhkan: ['0.4.10', '0.4.11', '0.4.13', '0.4.14', '0.4.15', '0.4.16', '0.4.17', '0.4.18', '0.4.19', '0.4.20', '0.4.21', '0.4.22', '0.4.23', '0.4.24', '0.4.25', '0.4.26', '0.4.4', '0.4.8', '0.5.0', '0.5.10', '0.5.11', '0.5.12', '0.5.15', '0.5.16', '0.5.17', '0.5.2', '0.5.4', '0.5.5', '0.5.6', '0.5.7', '0.5.8', '0.5.9', '0.6.0', '0.6.10', '0.6.11', '0.6.12', '0.6.2', '0.6.3', '0.6.4', '0.6.5', '0.6.6', '0.6.7', '0.6.8', '0.6.9', '0.7.0', '0.7.1', '0.7.2', '0.7.3', '0.7.4', '0.7.5', '0.7.6', '0.8.0', '0.8.1', '0.8.10', '0.8.2', '0.8.27', '0.8.3', '0.8.4', '0.8.6', '0.8.7', '0.8.9']


In [ ]:
from solc_select import solc_select
gagal = []
for i, v in enumerate(versi, 1):
    try:
        solc_select.install_artifacts([str(v)])
        print(f'[{i}/{len(versi)}] OK {v}')
    except Exception as e:
        gagal.append(v)
        print(f'[{i}/{len(versi)}] GAGAL {v}: {str(e)[:60]}')
print('gagal:', gagal)

Installing solc '0.4.10'...
Version '0.4.10' installed.
[1/61] OK 0.4.10
Installing solc '0.4.11'...
Version '0.4.11' installed.
[2/61] OK 0.4.11
Installing solc '0.4.13'...
Version '0.4.13' installed.
[3/61] OK 0.4.13
Installing solc '0.4.14'...
Version '0.4.14' installed.
[4/61] OK 0.4.14
Installing solc '0.4.15'...
Version '0.4.15' installed.
[5/61] OK 0.4.15
Installing solc '0.4.16'...
Version '0.4.16' installed.
[6/61] OK 0.4.16
Installing solc '0.4.17'...
Version '0.4.17' installed.
[7/61] OK 0.4.17
Installing solc '0.4.18'...
Version '0.4.18' installed.
[8/61] OK 0.4.18
Installing solc '0.4.19'...
Version '0.4.19' installed.
[9/61] OK 0.4.19
Installing solc '0.4.20'...
Version '0.4.20' installed.
[10/61] OK 0.4.20
Installing solc '0.4.21'...
Version '0.4.21' installed.
[11/61] OK 0.4.21
Installing solc '0.4.22'...
Version '0.4.22' installed.
[12/61] OK 0.4.22
Installing solc '0.4.23'...
Version '0.4.23' installed.
[13/61] OK 0.4.23
Installing solc '0.4.24'...
Version '0.4.24' in

In [ ]:
src = str(link / 'artifacts')
dst = f'{D}/solc_binaries/artifacts'
!cp -r "{src}/." "{dst}/"
print('tersalin:', len(os.listdir(dst)), 'item')

tersalin: 61 item


In [ ]:
import os
dst = f'{D}/solc_binaries/artifacts'
isi = sorted(os.listdir(dst))
print(f'{len(isi)} versi di Drive')
print(isi[:5])

# cek satu binary benar-benar ada isinya
contoh = os.path.join(dst, isi[0])
for f in os.listdir(contoh):
    p = os.path.join(contoh, f)
    print(f'  {f}: {os.path.getsize(p)/1024/1024:.1f} MB')

61 versi di Drive
['solc-0.4.10', 'solc-0.4.11', 'solc-0.4.13', 'solc-0.4.14', 'solc-0.4.15']
  solc-0.4.10: 13.9 MB


In [ ]:
import os
os.makedirs('/content/data/raw', exist_ok=True)
!unzip -q -o /content/drive/MyDrive/PROJECT/data/raw/dataset_final.zip -d /content/data/raw/

BASE = '/content/data/raw/dataset_penelitian'
KARANTINA = '/content/data/raw/karantina'
AMAN = f'{BASE}/3_expert_benchmark/aman'
RENTAN_BM = f'{BASE}/3_expert_benchmark/rentan'
MANIFEST3 = f'{D}/data/processed/manifest_final.csv'

for root, dirs, files in os.walk(BASE):
    sol = [f for f in files if f.endswith('.sol')]
    if sol:
        print(f'{root.replace(BASE, ""):36} {len(sol)}')

/0_normal                            500
/3_expert_benchmark/rentan           8
/3_expert_benchmark/aman             8
/1_reentrancy                        244


In [ ]:
import os
dst = f'{D}/solc_binaries/artifacts'
isi = sorted(os.listdir(dst))
print(f'{len(isi)} versi di Drive')
print(isi[:5])

contoh = os.path.join(dst, isi[0])
for f in os.listdir(contoh):
    p = os.path.join(contoh, f)
    print(f'  {f}: {os.path.getsize(p)/1024/1024:.1f} MB')

61 versi di Drive
['solc-0.4.10', 'solc-0.4.11', 'solc-0.4.13', 'solc-0.4.14', 'solc-0.4.15']
  solc-0.4.10: 13.9 MB


In [ ]:
import re, pandas as pd
from packaging.specifiers import SpecifierSet
from packaging.version import Version
from solc_select import solc_select

terpasang = sorted([Version(v) for v in solc_select.installed_versions()])
print(f'{len(terpasang)} versi terbaca solc-select')

def get_pragma(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    m = re.search(r'pragma\s+solidity\s*([^;]+);', content)
    return m.group(1).strip() if m else "TIDAK_DITEMUKAN"

def resolve_solc_version_v2(pragma_str):
    if not pragma_str or str(pragma_str) == 'TIDAK_DITEMUKAN' or pd.isna(pragma_str):
        return None
    s = str(pragma_str).replace('^', '~=').strip()
    try:
        spec = SpecifierSet(','.join(p.strip() for p in s.split() if p.strip()))
        cocok = [v for v in terpasang if v in spec]
        if cocok:
            return str(max(cocok))
    except Exception:
        pass
    m = re.search(r'(\d+\.\d+\.\d+)', str(pragma_str))
    return m.group(1) if m else None

MANIFEST3 = f'{D}/data/processed/manifest_final.csv'
df_final = pd.read_csv(MANIFEST3)
print(f'{len(df_final)} baris manifest')

61 versi terbaca solc-select
760 baris manifest


In [ ]:
import subprocess
from slither import Slither

fp = f'{BASE}/3_expert_benchmark/rentan/simple_dao.sol'
ver = resolve_solc_version_v2(get_pragma(fp))
subprocess.run(['solc-select', 'use', ver], capture_output=True)
print(f'solc {ver}\n')

sl = Slither(fp)
for c in sl.contracts:
    print(f'=== contract {c.name} ===')
    for fn in c.functions:
        print(f'\n  fungsi {fn.name}  ({len(fn.nodes)} node)')
        for n in fn.nodes:
            ext = len(n.low_level_calls) + len(n.high_level_calls)
            sw = len(n.state_variables_written)
            sons = [m.node_id for m in n.sons]
            print(f'    id={n.node_id:2} type={str(n.type):20} call={ext} sstore={sw} sons={sons}')
            print(f'         expr: {str(n.expression)[:70]}')

solc 0.4.26

=== contract SimpleDAO ===

  fungsi donate  (2 node)
    id= 0 type=NodeType.ENTRYPOINT  call=0 sstore=0 sons=[1]
         expr: None
    id= 1 type=NodeType.EXPRESSION  call=0 sstore=1 sons=[]
         expr: credit[to] += msg.value

  fungsi withdraw  (5 node)
    id= 0 type=NodeType.ENTRYPOINT  call=0 sstore=0 sons=[1]
         expr: None
    id= 1 type=NodeType.IF          call=0 sstore=0 sons=[2, 4]
         expr: credit[msg.sender] >= amount
    id= 2 type=NodeType.VARIABLE    call=1 sstore=0 sons=[3]
         expr: res = msg.sender.call.value(amount)()
    id= 3 type=NodeType.EXPRESSION  call=0 sstore=1 sons=[4]
         expr: credit[msg.sender] -= amount
    id= 4 type=NodeType.ENDIF       call=0 sstore=0 sons=[]
         expr: None

  fungsi queryCredit  (2 node)
    id= 0 type=NodeType.ENTRYPOINT  call=0 sstore=0 sons=[1]
         expr: None
    id= 1 type=NodeType.RETURN      call=0 sstore=0 sons=[]
         expr: credit[to]


# Fungsi Ekstraksi

In [ ]:
import os, json, subprocess, time
from slither import Slither
import pandas as pd

OUT_CFG = f'{D}/data/processed/cfg'
os.makedirs(OUT_CFG, exist_ok=True)

TIPE_NODE = [
    'ENTRYPOINT', 'EXPRESSION', 'RETURN', 'IF', 'ENDIF', 'VARIABLE',
    'IFLOOP', 'ENDLOOP', 'STARTLOOP', 'THROW', 'BREAK', 'CONTINUE',
    'ASSEMBLY', 'PLACEHOLDER', 'OTHER_ENTRYPOINT',
]

def ekstrak_cfg(fp):
    """Bangun satu graf gabungan untuk seluruh fungsi di satu file .sol."""
    sl = Slither(fp)

    peta = {}          # (contract, fungsi, node_id lama) -> id global
    nodes = []
    fungsi_entry = {}  # signature fungsi -> id global node ENTRYPOINT-nya

    # tahap 1: daftarkan semua node
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                gid = len(nodes)
                peta[(c.name, fn.full_name, n.node_id)] = gid
                tipe = str(n.type).replace('NodeType.', '')
                nodes.append({
                    'id': gid,
                    'contract': c.name,
                    'fungsi': fn.full_name,
                    'tipe': tipe,
                    'n_call_low': len(n.low_level_calls),
                    'n_call_high': len(n.high_level_calls),
                    'n_internal': len(n.internal_calls),
                    'n_sstore': len(n.state_variables_written),
                    'n_sload': len(n.state_variables_read),
                    'n_irs': len(n.irs),
                    'expr': str(n.expression)[:200] if n.expression else '',
                })
                if n.node_id == 0:
                    fungsi_entry[(c.name, fn.full_name)] = gid

    # tahap 2: sisi intra-fungsi
    edges = []
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                a = peta[(c.name, fn.full_name, n.node_id)]
                for m in n.sons:
                    kunci = (c.name, fn.full_name, m.node_id)
                    if kunci in peta:
                        edges.append([a, peta[kunci], 0])   # 0 = alur eksekusi

    # tahap 3: sisi antar-fungsi (dari titik pemanggilan ke entrypoint)
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                a = peta[(c.name, fn.full_name, n.node_id)]
                for ic in n.internal_calls:
                    target = getattr(ic, 'function', ic)
                    nama = getattr(target, 'full_name', None)
                    if nama and (c.name, nama) in fungsi_entry:
                        edges.append([a, fungsi_entry[(c.name, nama)], 1])  # 1 = panggilan

    return {'nodes': nodes, 'edges': edges,
            'n_node': len(nodes), 'n_edge': len(edges),
            'n_contract': len(sl.contracts)}

In [ ]:
os.chdir('/content')
df_ok = df_final[df_final.status == 'OK'].copy()
df_ok = df_ok.sort_values('solc')
print(f'{len(df_ok)} kontrak akan diekstraksi')

sudah = {f[:-5] for f in os.listdir(OUT_CFG) if f.endswith('.json')}
print(f'{len(sudah)} sudah ada, dilewati')

versi_aktif, berhasil, gagal = None, 0, []
mulai = time.time()

for i, (_, r) in enumerate(df_ok.iterrows(), 1):
    kunci = f"{r.folder.replace('/', '__')}__{r.nama_file[:-4]}"
    if kunci in sudah:
        continue
    try:
        if r.solc != versi_aktif:
            subprocess.run(['solc-select', 'use', str(r.solc)], capture_output=True, check=True)
            versi_aktif = r.solc
        g = ekstrak_cfg(r.path)
        g.update({'nama_file': r.nama_file, 'folder': r.folder,
                  'label': int(r.label), 'split': r.split, 'solc': str(r.solc)})
        with open(f'{OUT_CFG}/{kunci}.json', 'w') as f:
            json.dump(g, f)
        berhasil += 1
        if i % 25 == 0 or berhasil <= 5:
            lewat = time.time() - mulai
            print(f'[{i}/{len(df_ok)}] {r.nama_file[:26]} node={g["n_node"]} edge={g["n_edge"]} ({lewat/60:.1f} mnt)')
    except Exception as e:
        gagal.append((r.nama_file, str(e)[:100]))
        print(f'[{i}/{len(df_ok)}] GAGAL {r.nama_file[:26]}: {str(e)[:60]}')

print(f'\nSelesai. Berhasil {berhasil}, gagal {len(gagal)}, waktu {(time.time()-mulai)/60:.1f} menit')

760 kontrak akan diekstraksi
0 sudah ada, dilewati


[1/760] 0x01805590991328db7f1ef0e8 node=323 edge=248 (0.0 mnt)


[2/760] 0x0000000000b3F879cb30FE24 node=147 edge=134 (0.0 mnt)
[3/760] 0x01d9f4d104668cdc0b6d13c4 node=633 edge=655 (0.1 mnt)


[4/760] 0x001ea8150f4965195e10e5b5 node=60 edge=40 (0.1 mnt)


[5/760] 0x01417fea37677b581c1237b7 node=56 edge=43 (0.1 mnt)


[25/760] 0x0132bfca599d0b5b12f9160f node=115 edge=85 (0.4 mnt)


... (26 outputs left out)

[700/760] 0x01273fb613e983431cbe3a20 node=617 edge=633 (18.5 mnt)


[725/760] 0x0027903278b92d68df9a8a17 node=473 edge=470 (19.0 mnt)


[750/760] 0x005f8205ddcbb28b1295d5c2 node=291 edge=291 (19.4 mnt)



Selesai. Berhasil 760, gagal 0, waktu 19.6 menit


In [ ]:
import json, os

def baca(kunci):
    with open(f'{OUT_CFG}/{kunci}.json') as f:
        return json.load(f)

a = baca('3_expert_benchmark__rentan__simple_dao')
b = baca('3_expert_benchmark__aman__simple_dao_fixed')

for nama, g in [('RENTAN', a), ('AMAN  ', b)]:
    print(f'\n=== {nama}  node={g["n_node"]} edge={g["n_edge"]} ===')
    for n in g['nodes']:
        if n['fungsi'].startswith('withdraw'):
            keluar = [e[1] for e in g['edges'] if e[0] == n['id']]
            print(f'  id={n["id"]:2} {n["tipe"]:12} call={n["n_call_low"]+n["n_call_high"]} '
                  f'sstore={n["n_sstore"]} -> {keluar}  | {n["expr"][:45]}')


=== RENTAN  node=9 edge=7 ===
  id= 2 ENTRYPOINT   call=0 sstore=0 -> [3]  | 
  id= 3 IF           call=0 sstore=0 -> [4, 6]  | credit[msg.sender] >= amount
  id= 4 VARIABLE     call=1 sstore=0 -> [5]  | res = msg.sender.call.value(amount)()
  id= 5 EXPRESSION   call=0 sstore=1 -> [6]  | credit[msg.sender] -= amount
  id= 6 ENDIF        call=0 sstore=0 -> []  | 

=== AMAN    node=9 edge=7 ===
  id= 2 ENTRYPOINT   call=0 sstore=0 -> [3]  | 
  id= 3 IF           call=0 sstore=0 -> [4, 6]  | credit[msg.sender] >= amount
  id= 4 EXPRESSION   call=0 sstore=1 -> [5]  | credit[msg.sender] -= amount
  id= 5 VARIABLE     call=1 sstore=0 -> [6]  | res = msg.sender.call.value(amount)()
  id= 6 ENDIF        call=0 sstore=0 -> []  | 


In [ ]:
import pandas as pd

baris = []
for f in os.listdir(OUT_CFG):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_CFG}/{f}') as fh:
        g = json.load(fh)
    baris.append({'nama': g['nama_file'], 'folder': g['folder'], 'label': g['label'],
                  'split': g['split'], 'n_node': g['n_node'], 'n_edge': g['n_edge'],
                  'n_contract': g['n_contract']})

stat = pd.DataFrame(baris)
stat.to_csv(f'{D}/data/processed/statistik_cfg.csv', index=False)

print(f'{len(stat)} graf\n')
print(stat.groupby('split')[['n_node', 'n_edge']].describe().T)
print('\nDistribusi n_node:')
print(stat.n_node.quantile([.5, .75, .9, .95, .99, 1.0]))

760 graf

split         benchmark        train
n_node count  16.000000   744.000000
       mean    9.937500   479.154570
       std     2.434988  1057.043314
       min     6.000000     2.000000
       25%     9.000000    74.750000
       50%    10.000000   170.000000
       75%    11.000000   378.000000
       max    15.000000  6635.000000
n_edge count  16.000000   744.000000
       mean    7.500000   477.840054
       std     2.097618  1116.843607
       min     4.000000     1.000000
       25%     6.750000    67.500000
       50%     7.000000   154.500000
       75%    10.000000   346.500000
       max    11.000000  7396.000000

Distribusi n_node:
0.50     167.00
0.75     371.25
0.90     715.10
0.95    2198.75
0.99    5655.07
1.00    6635.00
Name: n_node, dtype: float64


In [ ]:
kecil = stat[(stat.split == 'train') & (stat.n_node < 10)].sort_values('n_node')
print(f'{len(kecil)} graf latih dengan node < 10')
print(kecil[['nama', 'folder', 'n_node', 'n_edge', 'n_contract']].head(15).to_string())

25 graf latih dengan node < 10
                                               nama    folder  n_node  n_edge  n_contract
652  0x000000000000cd17345801aa8147b8D3950260FF.sol  0_normal       2       1           2
651  0x0000000000a84fe7f5d858c8a22121c975ff0b42.sol  0_normal       2       1           1
122  0x0025336944da9e6c4554f350d1e133b107845fd3.sol  0_normal       3       2           1
499  0xc49eaf505de79f0457c085a31576edc5dc9fbad3.sol  0_normal       4       2           3
494  0xec476dd4dbf8dfa147244867c0b3b9a5f11d171c.sol  0_normal       4       2           3
506  0xa5725860d54a6e8d7c5561c1f1e648142eccb85e.sol  0_normal       4       2           3
487  0x089f0be0354f47d005e3871299cddfdfffdbe5e6.sol  0_normal       4       2           3
479  0x1a3de69a70a7d4e6fb6a207a9aaf8ee3a7cb2709.sol  0_normal       4       2           3
509  0xa0ed9ce70241e907c003a5fdfeef6b40c67a5370.sol  0_normal       4       2           3
398  0x019317a471ba529b59f288569b8885ca9a900fc7.sol  0_normal       5

In [ ]:
from collections import Counter
c = Counter()
for f in os.listdir(OUT_CFG):
    if f.endswith('.json'):
        with open(f'{OUT_CFG}/{f}') as fh:
            for n in json.load(fh)['nodes']:
                c[n['tipe']] += 1
total = sum(c.values())
for t, j in c.most_common():
    print(f'{t:22} {j:8}  {j/total*100:5.1f}%')

EXPRESSION               132021   37.0%
ENTRYPOINT                68229   19.1%
RETURN                    54645   15.3%
VARIABLE                  41015   11.5%
IF                        21161    5.9%
ENDIF                     18910    5.3%
OTHER_ENTRYPOINT           8253    2.3%
STARTLOOP                  2753    0.8%
ENDLOOP                    2753    0.8%
IFLOOP                     2750    0.8%
ASSEMBLY                   1849    0.5%
ENDASSEMBLY                1849    0.5%
BREAK                       204    0.1%
CONTINUE                     96    0.0%
CATCH                        89    0.0%
TRY                          40    0.0%
THROW                        33    0.0%


In [ ]:
import os, json, math
from collections import Counter

def hitung_idf(cfg_dir):
    """IDF per tipe node, dihitung sekali dari seluruh korpus."""
    df_tipe = Counter()
    n_dok = 0
    for f in os.listdir(cfg_dir):
        if not f.endswith('.json'):
            continue
        n_dok += 1
        with open(f'{cfg_dir}/{f}') as fh:
            tipe_ada = {n['tipe'] for n in json.load(fh)['nodes']}
        for t in tipe_ada:
            df_tipe[t] += 1
    return {t: math.log(n_dok / d) for t, d in df_tipe.items()}, n_dok


def wajib_simpan(n):
    """Node yang tidak boleh dipangkas apa pun skornya."""
    if n['n_call_low'] + n['n_call_high'] > 0:   # pemanggilan eksternal
        return True
    if n['n_sstore'] > 0:                        # penulisan state
        return True
    if n['tipe'] in ('IF', 'IFLOOP'):            # percabangan
        return True
    if n['tipe'] in ('ENTRYPOINT', 'OTHER_ENTRYPOINT'):  # titik sambung antar fungsi
        return True
    if n['n_internal'] > 0:                      # pemanggilan fungsi internal
        return True
    return False


def prune_graf(g, idf, ambang):
    """Pangkas node berskor rendah, lalu sambung ulang sisi yang terputus."""
    tf = Counter(n['tipe'] for n in g['nodes'])
    total = max(len(g['nodes']), 1)

    simpan = set()
    for n in g['nodes']:
        if wajib_simpan(n):
            simpan.add(n['id'])
            continue
        skor = (tf[n['tipe']] / total) * idf.get(n['tipe'], 0.0)
        if skor >= ambang:
            simpan.add(n['id'])

    # sambung ulang: lewati node yang dibuang di sepanjang jalur
    keluar = {}
    for a, b, t in g['edges']:
        keluar.setdefault(a, []).append((b, t))

    def telusuri(mulai, tipe_sisi, terlihat):
        """Cari node bertahan pertama dari titik yang dibuang."""
        hasil = []
        for b, t in keluar.get(mulai, []):
            if b in terlihat:
                continue
            terlihat.add(b)
            if b in simpan:
                hasil.append((b, tipe_sisi))
            else:
                hasil.extend(telusuri(b, tipe_sisi, terlihat))
        return hasil

    edges_baru = set()
    for a, b, t in g['edges']:
        if a not in simpan:
            continue
        if b in simpan:
            edges_baru.add((a, b, t))
        else:
            for c, tt in telusuri(b, t, {a, b}):
                edges_baru.add((a, c, tt))

    # penomoran ulang
    urut = sorted(simpan)
    peta = {lama: baru for baru, lama in enumerate(urut)}
    asal = {n['id']: n for n in g['nodes']}

    nodes_baru = []
    for lama in urut:
        n = dict(asal[lama])
        n['id'] = peta[lama]
        nodes_baru.append(n)

    hasil = dict(g)
    hasil['nodes'] = nodes_baru
    hasil['edges'] = [[peta[a], peta[b], t] for a, b, t in sorted(edges_baru)]
    hasil['n_node'] = len(nodes_baru)
    hasil['n_edge'] = len(hasil['edges'])
    hasil['n_node_asli'] = g['n_node']
    hasil['n_edge_asli'] = g['n_edge']
    hasil['ambang_prune'] = ambang
    return hasil

In [ ]:
import sys, time
sys.setrecursionlimit(20000)

AMBANG = 0.02   # ubah nilai ini untuk mencoba konfigurasi lain
OUT_PRUNE = f'{D}/data/processed/cfg_prune_{AMBANG}'
os.makedirs(OUT_PRUNE, exist_ok=True)

idf, n_dok = hitung_idf(OUT_CFG)
print(f'IDF dari {n_dok} graf')
for t, v in sorted(idf.items(), key=lambda x: x[1])[:6]:
    print(f'  {t:20} idf={v:.3f}')

mulai = time.time()
ringkas = []
for i, f in enumerate(sorted(os.listdir(OUT_CFG)), 1):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_CFG}/{f}') as fh:
        g = json.load(fh)
    p = prune_graf(g, idf, AMBANG)
    with open(f'{OUT_PRUNE}/{f}', 'w') as fh:
        json.dump(p, fh)
    ringkas.append({'nama': g['nama_file'], 'split': g['split'], 'label': g['label'],
                    'asli': g['n_node'], 'prune': p['n_node']})

import pandas as pd
rk = pd.DataFrame(ringkas)
rk['rasio'] = rk.prune / rk.asli
print(f'\n{len(rk)} graf dipangkas dalam {time.time()-mulai:.1f} detik')
print(f'node total: {rk.asli.sum()} -> {rk.prune.sum()} ({rk.prune.sum()/rk.asli.sum()*100:.1f}%)')
print(rk.groupby('split')[['asli', 'prune', 'rasio']].mean())

IDF dari 760 graf
  ENTRYPOINT           idf=0.001
  EXPRESSION           idf=0.023
  RETURN               idf=0.035
  VARIABLE             idf=0.101
  IF                   idf=0.173
  ENDIF                idf=0.200

760 graf dipangkas dalam 15.9 detik
node total: 356650 -> 245096 (68.7%)
                asli      prune     rasio
split                                    
benchmark    9.93750    8.68750  0.885927
train      479.15457  329.24328  0.698015


In [ ]:
def baca_p(kunci):
    with open(f'{OUT_PRUNE}/{kunci}.json') as f:
        return json.load(f)

pasangan = [
    ('simple_dao', '3_expert_benchmark__rentan__simple_dao', '3_expert_benchmark__aman__simple_dao_fixed'),
    ('reentrance', '3_expert_benchmark__rentan__reentrance', '3_expert_benchmark__aman__reentrance_fixed'),
    ('etherstore', '3_expert_benchmark__rentan__etherstore', '3_expert_benchmark__aman__etherstore_fixed'),
    ('etherbank',  '3_expert_benchmark__rentan__etherbank',  '3_expert_benchmark__aman__etherbank_fixed'),
]

for nama, kr, ka in pasangan:
    r, a = baca_p(kr), baca_p(ka)
    sama = (r['n_node'] == a['n_node'] and
            sorted(map(tuple, r['edges'])) == sorted(map(tuple, a['edges'])))
    print(f'{nama:14} rentan {r["n_node"]}n/{r["n_edge"]}e   aman {a["n_node"]}n/{a["n_edge"]}e   '
          f'{"IDENTIK - BAHAYA" if sama else "berbeda - OK"}')

# detail satu pasangan
print('\n--- simple_dao setelah prune ---')
for label, g in [('RENTAN', baca_p(pasangan[0][1])), ('AMAN  ', baca_p(pasangan[0][2]))]:
    print(f'\n{label}')
    for n in g['nodes']:
        if n['fungsi'].startswith('withdraw'):
            keluar = [e[1] for e in g['edges'] if e[0] == n['id']]
            print(f'  id={n["id"]:2} {n["tipe"]:12} call={n["n_call_low"]+n["n_call_high"]} '
                  f'sstore={n["n_sstore"]} -> {keluar}')

simple_dao     rentan 8n/6e   aman 8n/6e   IDENTIK - BAHAYA
reentrance     rentan 10n/7e   aman 10n/7e   IDENTIK - BAHAYA
etherstore     rentan 10n/7e   aman 10n/7e   IDENTIK - BAHAYA
etherbank      rentan 8n/5e   aman 8n/5e   berbeda - OK

--- simple_dao setelah prune ---

RENTAN
  id= 2 ENTRYPOINT   call=0 sstore=0 -> [3]
  id= 3 IF           call=0 sstore=0 -> [4, 6]
  id= 4 VARIABLE     call=1 sstore=0 -> [5]
  id= 5 EXPRESSION   call=0 sstore=1 -> [6]
  id= 6 ENDIF        call=0 sstore=0 -> []

AMAN  
  id= 2 ENTRYPOINT   call=0 sstore=0 -> [3]
  id= 3 IF           call=0 sstore=0 -> [4, 6]
  id= 4 EXPRESSION   call=0 sstore=1 -> [5]
  id= 5 VARIABLE     call=1 sstore=0 -> [6]
  id= 6 ENDIF        call=0 sstore=0 -> []


In [ ]:
def sidik(g):
    """Ringkas tiap node jadi (tipe, ada call, ada sstore), diurutkan per id."""
    return [(n['tipe'], int(n['n_call_low'] + n['n_call_high'] > 0), int(n['n_sstore'] > 0))
            for n in sorted(g['nodes'], key=lambda x: x['id'])]

print(f'{"pasangan":16} {"node":>10}  status')
for nama, kr, ka in pasangan:
    r, a = baca_p(kr), baca_p(ka)
    sama_edge = sorted(map(tuple, r['edges'])) == sorted(map(tuple, a['edges']))
    sama_fitur = sidik(r) == sidik(a)
    if sama_edge and sama_fitur:
        status = 'IDENTIK TOTAL - BAHAYA'
    elif sama_edge:
        status = 'topologi sama, fitur beda - OK'
    else:
        status = 'topologi beda - OK'
    print(f'{nama:16} {r["n_node"]:>4}/{a["n_node"]:<4}  {status}')

pasangan               node  status
simple_dao          8/8     topologi sama, fitur beda - OK
reentrance         10/10    topologi sama, fitur beda - OK
etherstore         10/10    topologi sama, fitur beda - OK
etherbank           8/8     topologi beda - OK


# Konversi ke PyTorch Geometric

In [ ]:
import os, json, math, torch
from torch_geometric.data import Data

TIPE = ['EXPRESSION', 'ENTRYPOINT', 'RETURN', 'VARIABLE', 'IF', 'ENDIF',
        'OTHER_ENTRYPOINT', 'STARTLOOP', 'ENDLOOP', 'IFLOOP', 'ASSEMBLY',
        'ENDASSEMBLY', 'BREAK', 'CONTINUE', 'CATCH', 'TRY', 'THROW']
TIPE_IDX = {t: i for i, t in enumerate(TIPE)}
DIM = len(TIPE) + 5

def vektor_node(n):
    v = [0.0] * DIM
    idx = TIPE_IDX.get(n['tipe'])
    if idx is not None:
        v[idx] = 1.0
    o = len(TIPE)
    v[o + 0] = math.log1p(n['n_call_low'])
    v[o + 1] = math.log1p(n['n_call_high'])
    v[o + 2] = math.log1p(n['n_sstore'])
    v[o + 3] = math.log1p(n['n_sload'])
    v[o + 4] = math.log1p(n['n_irs'])
    return v

def ke_pyg(g):
    nodes = sorted(g['nodes'], key=lambda x: x['id'])
    x = torch.tensor([vektor_node(n) for n in nodes], dtype=torch.float)

    if g['edges']:
        ei = torch.tensor([[e[0] for e in g['edges']],
                           [e[1] for e in g['edges']]], dtype=torch.long)
        et = torch.tensor([e[2] for e in g['edges']], dtype=torch.long)
    else:
        ei = torch.empty((2, 0), dtype=torch.long)
        et = torch.empty((0,), dtype=torch.long)

    return Data(x=x, edge_index=ei, edge_type=et,
                y=torch.tensor([g['label']], dtype=torch.long),
                nama=g['nama_file'], split=g['split'], folder=g['folder'])

In [ ]:
SUMBER = OUT_PRUNE          # ganti ke OUT_CFG untuk versi tanpa pruning
NAMA_SET = os.path.basename(SUMBER)
OUT_PYG = f'{D}/data/processed/pyg'
os.makedirs(OUT_PYG, exist_ok=True)

train, bench, kosong = [], [], []
for f in sorted(os.listdir(SUMBER)):
    if not f.endswith('.json'):
        continue
    with open(f'{SUMBER}/{f}') as fh:
        g = json.load(fh)
    d = ke_pyg(g)
    if d.num_nodes == 0 or d.edge_index.size(1) == 0:
        kosong.append(g['nama_file'])
        continue
    (train if g['split'] == 'train' else bench).append(d)

torch.save({'train': train, 'benchmark': bench, 'dim': DIM, 'tipe': TIPE},
           f'{OUT_PYG}/{NAMA_SET}.pt')

print(f'train     : {len(train)} graf')
print(f'benchmark : {len(bench)} graf')
print(f'dilewati  : {len(kosong)} graf tanpa node/edge')
print(f'dimensi   : {DIM}')

import collections
print('\nsebaran label train    :', collections.Counter(int(d.y) for d in train))
print('sebaran label benchmark:', collections.Counter(int(d.y) for d in bench))
print(f'\ntersimpan: {OUT_PYG}/{NAMA_SET}.pt')

train     : 742 graf
benchmark : 16 graf
dilewati  : 2 graf tanpa node/edge
dimensi   : 22

sebaran label train    : Counter({0: 498, 1: 244})
sebaran label benchmark: Counter({0: 8, 1: 8})

tersimpan: /content/drive/MyDrive/PROJECT/data/processed/pyg/cfg_prune_0.02.pt


In [ ]:
paket = torch.load(f'{OUT_PYG}/cfg_prune_0.02.pt', weights_only=False)
bench = paket['benchmark']

cari = {d.nama: d for d in bench}
r = cari['simple_dao.sol']
a = cari['simple_dao_fixed.sol']

o = len(TIPE)
print('kolom: [17]=call_low [18]=call_high [19]=sstore [20]=sload [21]=n_irs\n')
for label, d in [('RENTAN', r), ('AMAN  ', a)]:
    print(f'{label}  x={tuple(d.x.shape)}  edge={d.edge_index.size(1)}')
    for i in range(d.num_nodes):
        tipe = TIPE[int(d.x[i, :o].argmax())] if d.x[i, :o].sum() > 0 else '?'
        num = [round(float(v), 2) for v in d.x[i, o:]]
        tujuan = d.edge_index[1][d.edge_index[0] == i].tolist()
        print(f'  {i:2} {tipe:12} {num} -> {tujuan}')
    print()

print('x identik?', torch.equal(r.x, a.x))
print('edge identik?', torch.equal(r.edge_index, a.edge_index))

kolom: [17]=call_low [18]=call_high [19]=sstore [20]=sload [21]=n_irs

RENTAN  x=(8, 22)  edge=6
   0 ENTRYPOINT   [0.0, 0.0, 0.0, 0.0, 0.0] -> [1]
   1 EXPRESSION   [0.0, 0.0, 0.69, 0.69, 1.1] -> []
   2 ENTRYPOINT   [0.0, 0.0, 0.0, 0.0, 0.0] -> [3]
   3 IF           [0.0, 0.0, 0.0, 0.69, 1.39] -> [4, 6]
   4 VARIABLE     [0.69, 0.0, 0.0, 0.0, 1.1] -> [5]
   5 EXPRESSION   [0.0, 0.0, 0.69, 0.69, 1.1] -> [6]
   6 ENDIF        [0.0, 0.0, 0.0, 0.0, 0.0] -> []
   7 ENTRYPOINT   [0.0, 0.0, 0.0, 0.0, 0.0] -> []

AMAN    x=(8, 22)  edge=6
   0 ENTRYPOINT   [0.0, 0.0, 0.0, 0.0, 0.0] -> [1]
   1 EXPRESSION   [0.0, 0.0, 0.69, 0.69, 1.1] -> []
   2 ENTRYPOINT   [0.0, 0.0, 0.0, 0.0, 0.0] -> [3]
   3 IF           [0.0, 0.0, 0.0, 0.69, 1.39] -> [4, 6]
   4 EXPRESSION   [0.0, 0.0, 0.69, 0.69, 1.1] -> [5]
   5 VARIABLE     [0.69, 0.0, 0.0, 0.0, 1.1] -> [6]
   6 ENDIF        [0.0, 0.0, 0.0, 0.0, 0.0] -> []
   7 ENTRYPOINT   [0.0, 0.0, 0.0, 0.0, 0.0] -> []

x identik? False
edge identik? True


In [ ]:
VERSI TANPA PRUNING

In [ ]:
SUMBER = OUT_CFG            # versi tanpa pruning
NAMA_SET = os.path.basename(SUMBER)
OUT_PYG = f'{D}/data/processed/pyg'
os.makedirs(OUT_PYG, exist_ok=True)

train, bench, kosong = [], [], []
for f in sorted(os.listdir(SUMBER)):
    if not f.endswith('.json'):
        continue
    with open(f'{SUMBER}/{f}') as fh:
        g = json.load(fh)
    d = ke_pyg(g)
    if d.num_nodes == 0 or d.edge_index.size(1) == 0:
        kosong.append(g['nama_file'])
        continue
    (train if g['split'] == 'train' else bench).append(d)

torch.save({'train': train, 'benchmark': bench, 'dim': DIM, 'tipe': TIPE},
           f'{OUT_PYG}/{NAMA_SET}.pt')

print(f'train     : {len(train)} graf')
print(f'benchmark : {len(bench)} graf')
print(f'dilewati  : {len(kosong)} graf tanpa node/edge')

import collections
print('\nsebaran label train    :', collections.Counter(int(d.y) for d in train))
print('sebaran label benchmark:', collections.Counter(int(d.y) for d in bench))
print(f'\ntersimpan: {OUT_PYG}/{NAMA_SET}.pt')
print('\nisi folder pyg:', os.listdir(OUT_PYG))

train     : 744 graf
benchmark : 16 graf
dilewati  : 0 graf tanpa node/edge

sebaran label train    : Counter({0: 500, 1: 244})
sebaran label benchmark: Counter({0: 8, 1: 8})

tersimpan: /content/drive/MyDrive/PROJECT/data/processed/pyg/cfg.pt

isi folder pyg: ['cfg_prune_0.02.pt', 'cfg.pt']


# SET UP AWAL UNTUK MULTISCALE GAT

In [ ]:
# === SETUP SESI (jalankan sekali di awal) ===
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch_geometric

import os, json, math, torch
D = '/content/drive/MyDrive/PROJECT'
OUT_PYG = f'{D}/data/processed/pyg'

TIPE = ['EXPRESSION', 'ENTRYPOINT', 'RETURN', 'VARIABLE', 'IF', 'ENDIF',
        'OTHER_ENTRYPOINT', 'STARTLOOP', 'ENDLOOP', 'IFLOOP', 'ASSEMBLY',
        'ENDASSEMBLY', 'BREAK', 'CONTINUE', 'CATCH', 'TRY', 'THROW']
DIM = len(TIPE) + 5

paket = torch.load(f'{OUT_PYG}/cfg_prune_0.02.pt', weights_only=False)
print('train    :', len(paket['train']))
print('benchmark:', len(paket['benchmark']))
print('dimensi  :', paket['dim'])
print('isi folder pyg:', os.listdir(OUT_PYG))

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.8 MB/s eta 0:00:00
train    : 742
benchmark: 16
dimensi  : 22
isi folder pyg: ['cfg_prune_0.02.pt', 'cfg.pt']


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, SAGPooling, global_mean_pool


class MultiScaleGAT(nn.Module):
    """Graph Branch (3 lapis GAT) -> Self-Attention Pooling -> Classifier MLP."""

    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden

        # Lapisan 1: 8 head, tiap head menghasilkan h//8 agar total tetap h
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        # Lapisan 2 dan 3: 4 head
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)

        # koneksi sisa antar lapisan, mencegah informasi awal hilang
        self.proj = nn.Linear(dim_input, h)

        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)

        # Self-Attention Pooling: N node -> 1 vektor kontrak
        self.pool = SAGPooling(h, ratio=0.5)

        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        """Hasilkan satu vektor per graf. Dipisah agar bisa dipakai SHAP."""
        h0 = self.proj(x)

        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)

        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)

        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)

        x_p, ei_p, _, batch_p, _, _ = self.pool(h3, edge_index, batch=batch)
        return global_mean_pool(x_p, batch_p)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

In [ ]:
paket = torch.load(f'{OUT_PYG}/cfg_prune_0.02.pt', weights_only=False)
from torch_geometric.loader import DataLoader

model = MultiScaleGAT(dim_input=paket['dim'], dim_hidden=512)
n_param = sum(p.numel() for p in model.parameters())
print(f'jumlah parameter: {n_param:,}')

loader = DataLoader(paket['train'][:8], batch_size=4)
batch = next(iter(loader))
out = model(batch)
print('input batch :', batch.x.shape, batch.edge_index.shape)
print('output      :', out.shape)
print('contoh logit:', out[0].detach().numpy().round(3))

jumlah parameter: 630,084
input batch : torch.Size([279, 22]) torch.Size([2, 226])
output      : torch.Size([4, 2])
contoh logit: [ 0.065 -0.062]


K FOLD

In [ ]:
import numpy as np, torch, time, copy
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEV)

def bobot_kelas(dataset):
    y = np.array([int(d.y) for d in dataset])
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    w = torch.tensor([len(y) / (2 * n0), len(y) / (2 * n1)], dtype=torch.float)
    return w.to(DEV)

@torch.no_grad()
def evaluasi(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    for b in loader:
        b = b.to(DEV)
        logit = model(b)
        p = torch.softmax(logit, dim=1)[:, 1]
        preds += logit.argmax(1).cpu().tolist()
        probs += p.cpu().tolist()
        ys += b.y.cpu().tolist()
    m = {
        'acc': accuracy_score(ys, preds),
        'prec': precision_score(ys, preds, zero_division=0),
        'rec': recall_score(ys, preds, zero_division=0),
        'f1': f1_score(ys, preds, zero_division=0),
    }
    m['delta_pr'] = abs(m['prec'] - m['rec'])
    try:
        m['auc'] = roc_auc_score(ys, probs)
    except Exception:
        m['auc'] = float('nan')
    m['cm'] = confusion_matrix(ys, preds).tolist()
    return m


def latih_satu_fold(tr, va, dim_hidden=512, max_epoch=50, sabar=15,
                    lr=1e-3, batch_size=32, verbose=True):
    model = MultiScaleGAT(dim_input=DIM, dim_hidden=dim_hidden).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss(weight=bobot_kelas(tr))

    ld_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    ld_va = DataLoader(va, batch_size=batch_size)

    f1_terbaik, bobot_terbaik, mandek = -1, None, 0
    riwayat = []

    for ep in range(1, max_epoch + 1):
        model.train()
        total = 0
        for b in ld_tr:
            b = b.to(DEV)
            opt.zero_grad()
            loss = lossf(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss) * b.num_graphs

        m_tr = evaluasi(model, ld_tr)
        m_va = evaluasi(model, ld_va)
        riwayat.append({'epoch': ep, 'loss': total / len(tr),
                        'f1_train': m_tr['f1'], 'f1_val': m_va['f1']})

        if m_va['f1'] > f1_terbaik:
            f1_terbaik = m_va['f1']
            bobot_terbaik = copy.deepcopy(model.state_dict())
            mandek = 0
        else:
            mandek += 1

        if verbose and (ep % 5 == 0 or ep == 1):
            print(f'    ep{ep:3} loss={total/len(tr):.4f} '
                  f'f1_tr={m_tr["f1"]:.3f} f1_va={m_va["f1"]:.3f}')

        if mandek >= sabar:
            if verbose:
                print(f'    early stopping di epoch {ep}')
            break

    model.load_state_dict(bobot_terbaik)
    return model, riwayat, evaluasi(model, ld_va)

device: cuda


In [ ]:
DATASET = 'cfg_prune_0.02'
paket = torch.load(f'{OUT_PYG}/{DATASET}.pt', weights_only=False)
data = paket['train']
DIM = paket['dim']
y = np.array([int(d.y) for d in data])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
hasil, model_fold, riwayat_fold = [], [], []
mulai = time.time()

for k, (i_tr, i_va) in enumerate(skf.split(np.zeros(len(y)), y), 1):
    print(f'\n=== Fold {k}/5  (train {len(i_tr)}, val {len(i_va)}) ===')
    tr = [data[i] for i in i_tr]
    va = [data[i] for i in i_va]
    model, riwayat, m = latih_satu_fold(tr, va)
    m['fold'] = k
    hasil.append(m)
    model_fold.append(model)
    riwayat_fold.append(riwayat)
    print(f'  F1={m["f1"]:.4f} Acc={m["acc"]:.4f} P={m["prec"]:.4f} '
          f'R={m["rec"]:.4f} AUC={m["auc"]:.4f}')

import pandas as pd
hf = pd.DataFrame(hasil)
print(f'\n=== Rata-rata 5 fold ({(time.time()-mulai)/60:.1f} menit) ===')
print(hf[['acc','prec','rec','f1','delta_pr','auc']].agg(['mean','std']).round(4))

os.makedirs(f'{D}/hasil', exist_ok=True)
hf.to_csv(f'{D}/hasil/kfold_{DATASET}.csv', index=False)
torch.save({'state_dicts': [m.state_dict() for m in model_fold],
            'riwayat': riwayat_fold, 'hasil': hasil, 'dim_hidden': 512},
           f'{D}/hasil/model_{DATASET}.pt')
print('tersimpan ke', f'{D}/hasil/')


=== Fold 1/5  (train 593, val 149) ===


/tmp/ipykernel_1206/2257142310.py:64: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total += float(loss) * b.num_graphs


    ep  1 loss=0.6525 f1_tr=0.683 f1_va=0.639
    ep  5 loss=0.4521 f1_tr=0.779 f1_va=0.727
    ep 10 loss=0.3312 f1_tr=0.815 f1_va=0.787
    ep 15 loss=0.2494 f1_tr=0.857 f1_va=0.748
    ep 20 loss=0.2806 f1_tr=0.883 f1_va=0.875
    ep 25 loss=0.2559 f1_tr=0.876 f1_va=0.854
    ep 30 loss=0.2195 f1_tr=0.876 f1_va=0.863
    ep 35 loss=0.2445 f1_tr=0.886 f1_va=0.848
    early stopping di epoch 35
  F1=0.8750 Acc=0.9195 P=0.8936 R=0.8571 AUC=0.9459

=== Fold 2/5  (train 593, val 149) ===
    ep  1 loss=0.6124 f1_tr=0.691 f1_va=0.710
    ep  5 loss=0.4464 f1_tr=0.717 f1_va=0.754
    ep 10 loss=0.3876 f1_tr=0.759 f1_va=0.762
    ep 15 loss=0.3733 f1_tr=0.771 f1_va=0.769
    ep 20 loss=0.3465 f1_tr=0.788 f1_va=0.788
    ep 25 loss=0.3667 f1_tr=0.506 f1_va=0.486
    ep 30 loss=0.3393 f1_tr=0.664 f1_va=0.634
    ep 35 loss=0.2859 f1_tr=0.833 f1_va=0.755
    early stopping di epoch 37
  F1=0.8317 Acc=0.8859 P=0.8077 R=0.8571 AUC=0.9345

=== Fold 3/5  (train 594, val 148) ===
    ep  1 loss=0.6

BENCHMARK

Problem Solve

In [ ]:
import torch, numpy as np, pandas as pd
from torch_geometric.loader import DataLoader

DATASET = 'cfg_prune_0.02'
paket = torch.load(f'{OUT_PYG}/{DATASET}.pt', weights_only=False)
simpan = torch.load(f'{D}/hasil/model_{DATASET}.pt', weights_only=False)
bench = paket['benchmark']
DIM = paket['dim']

# muat kelima model
models = []
for sd in simpan['state_dicts']:
    m = MultiScaleGAT(dim_input=DIM, dim_hidden=simpan['dim_hidden']).to(DEV)
    m.load_state_dict(sd)
    m.eval()
    models.append(m)

# prediksi tiap kontrak oleh tiap model
ld = DataLoader(bench, batch_size=16)
prob_semua = []
with torch.no_grad():
    for m in models:
        p = []
        for b in ld:
            b = b.to(DEV)
            p += torch.softmax(m(b), dim=1)[:, 1].cpu().tolist()
        prob_semua.append(p)
prob_semua = np.array(prob_semua)          # bentuk: 5 model x 16 kontrak
prob_rata = prob_semua.mean(axis=0)

tabel = pd.DataFrame({
    'nama': [d.nama for d in bench],
    'label': [int(d.y) for d in bench],
    'prob_rentan': prob_rata.round(3),
    'pred': (prob_rata >= 0.5).astype(int),
    'setuju_model': [f'{int((prob_semua[:, i] >= 0.5).sum())}/5' for i in range(len(bench))],
})
tabel['benar'] = tabel.label == tabel.pred
print(tabel.sort_values(['label', 'nama']).to_string(index=False))

                         nama  label  prob_rentan  pred setuju_model  benar
          etherbank_fixed.sol      0        0.002     0          0/5   True
         etherstore_fixed.sol      0        0.173     0          1/5   True
modifier_reentrancy_fixed.sol      0        0.022     0          0/5   True
         reentrance_fixed.sol      0        0.185     0          1/5   True
     reentrancy_dao_fixed.sol      0        0.308     0          2/5   True
reentrancy_insecure_fixed.sol      0        0.000     0          0/5   True
  reentrancy_simple_fixed.sol      0        0.002     0          0/5   True
         simple_dao_fixed.sol      0        0.004     0          0/5   True
                etherbank.sol      1        0.002     0          0/5  False
               etherstore.sol      1        0.074     0          0/5  False
      modifier_reentrancy.sol      1        0.039     0          0/5  False
               reentrance.sol      1        0.187     0          1/5  False
           r

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score

y, pr = tabel.label, tabel.pred
print(f'\nAcc={accuracy_score(y, pr):.4f}  P={precision_score(y, pr, zero_division=0):.4f}  '
      f'R={recall_score(y, pr, zero_division=0):.4f}  F1={f1_score(y, pr, zero_division=0):.4f}  '
      f'AUC={roc_auc_score(y, tabel.prob_rentan):.4f}')

print('\n=== Analisis per pasangan ===')
cari = dict(zip(tabel.nama, tabel.prob_rentan))
lolos = 0
for f in sorted(tabel[tabel.label == 1].nama):
    r = cari[f]
    a = cari.get(f.replace('.sol', '_fixed.sol'))
    ok = r > a
    lolos += ok
    print(f'{f:28} rentan={r:.3f}  aman={a:.3f}  {"OK, rentan lebih tinggi" if ok else "TERBALIK"}')
print(f'\n{lolos}/8 pasangan diurutkan dengan benar')

tabel.to_csv(f'{D}/hasil/benchmark_{DATASET}.csv', index=False)


Acc=0.5000  P=0.0000  R=0.0000  F1=0.0000  AUC=0.5391

=== Analisis per pasangan ===
etherbank.sol                rentan=0.002  aman=0.002  TERBALIK
etherstore.sol               rentan=0.074  aman=0.173  TERBALIK
modifier_reentrancy.sol      rentan=0.039  aman=0.022  OK, rentan lebih tinggi
reentrance.sol               rentan=0.187  aman=0.185  OK, rentan lebih tinggi
reentrancy_dao.sol           rentan=0.405  aman=0.308  OK, rentan lebih tinggi
reentrancy_insecure.sol      rentan=0.000  aman=0.000  TERBALIK
reentrancy_simple.sol        rentan=0.002  aman=0.002  TERBALIK
simple_dao.sol               rentan=0.043  aman=0.004  OK, rentan lebih tinggi

4/8 pasangan diurutkan dengan benar


In [ ]:
import numpy as np, torch
from torch_geometric.loader import DataLoader

train = paket['train']
ld = DataLoader(train, batch_size=64)
prob = []
with torch.no_grad():
    for b in ld:
        b = b.to(DEV)
        p = np.mean([torch.softmax(m(b), 1)[:, 1].cpu().numpy() for m in models], axis=0)
        prob += p.tolist()

ukuran = np.array([d.num_nodes for d in train])
label = np.array([int(d.y) for d in train])
prob = np.array(prob)

print('korelasi ukuran graf dengan prob rentan:', round(np.corrcoef(ukuran, prob)[0, 1], 3))
for lo, hi in [(0, 20), (20, 50), (50, 150), (150, 500), (500, 99999)]:
    m = (ukuran >= lo) & (ukuran < hi)
    if m.sum():
        print(f'node {lo:>4}-{hi:<5} n={m.sum():3}  rentan asli={label[m].mean():.2f}  '
              f'prob rata model={prob[m].mean():.2f}')

korelasi ukuran graf dengan prob rentan: 0.455
node    0-20    n= 54  rentan asli=0.06  prob rata model=0.06
node   20-50    n=125  rentan asli=0.04  prob rata model=0.07
node   50-150   n=246  rentan asli=0.11  prob rata model=0.17
node  150-500   n=240  rentan asli=0.58  prob rata model=0.60
node  500-99999 n= 77  rentan asli=0.90  prob rata model=0.83


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, roc_auc_score

X = np.array([[np.log1p(d.num_nodes), np.log1p(d.edge_index.size(1))] for d in train])
y = label

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr = LogisticRegression(class_weight='balanced')
pred = cross_val_predict(lr, X, y, cv=skf)
prob_lr = cross_val_predict(lr, X, y, cv=skf, method='predict_proba')[:, 1]

print(f'Baseline ukuran saja  F1={f1_score(y, pred):.4f}  AUC={roc_auc_score(y, prob_lr):.4f}')
print(f'Multi-scale GAT       F1=0.8397  AUC=0.9298')

Baseline ukuran saja  F1=0.7444  AUC=0.8816
Multi-scale GAT       F1=0.8397  AUC=0.9298


DATASET SESUAI PANJANGNYA

In [ ]:
import os
D = '/content/drive/MyDrive/PROJECT'
OUT_CFG = f'{D}/data/processed/cfg'
OUT_PRUNE = f'{D}/data/processed/cfg_prune_0.02'
OUT_PYG = f'{D}/data/processed/pyg'
BASE = '/content/data/raw/dataset_penelitian'

if not os.path.exists(BASE):
    os.makedirs('/content/data/raw', exist_ok=True)
    !unzip -q -o {D}/data/raw/dataset_final.zip -d /content/data/raw/

print('OUT_CFG ada :', os.path.exists(OUT_CFG), len(os.listdir(OUT_CFG)), 'file')
print('BASE ada    :', os.path.exists(BASE))

OUT_CFG ada : True 760 file
BASE ada    : True


In [ ]:
os.makedirs('/content/data/raw', exist_ok=True)
!unzip -q -o /content/drive/MyDrive/PROJECT/data/raw/dataset_final.zip -d /content/data/raw/
BASE = '/content/data/raw/dataset_penelitian'

In [ ]:
import numpy as np, pandas as pd, json, os

# ukuran graf reentrancy yang sudah valid
ren = []
for f in os.listdir(OUT_CFG):
    if not f.startswith('1_reentrancy__'):
        continue
    with open(f'{OUT_CFG}/{f}') as fh:
        g = json.load(fh)
    # panjang source code, sebagai penaksir saat unduh nanti
    p = f'{BASE}/1_reentrancy/{g["nama_file"]}' if 'BASE' in dir() else None
    n_char = os.path.getsize(p) if p and os.path.exists(p) else None
    ren.append({'nama': g['nama_file'], 'n_node': g['n_node'], 'n_char': n_char})

ren = pd.DataFrame(ren)
print(f'{len(ren)} kontrak reentrancy\n')
print('sebaran n_node:')
print(ren.n_node.describe().round(1))
print('\nkuantil n_node:')
print(ren.n_node.quantile([.1, .25, .5, .75, .9]).round(0))

if ren.n_char.notna().all():
    print('\nkorelasi n_char dengan n_node:', round(ren.n_char.corr(ren.n_node), 3))
    print('\nkuantil n_char (byte):')
    print(ren.n_char.quantile([.1, .25, .5, .75, .9]).round(0))
    ren.to_csv(f'{D}/data/processed/ukuran_reentrancy.csv', index=False)
    print('\ntersimpan ukuran_reentrancy.csv')
else:
    print('\ncatatan: file .sol tidak ditemukan di /content, perlu unzip dataset dulu')

244 kontrak reentrancy

sebaran n_node:
count     244.0
mean     1120.5
std      1649.5
min        10.0
25%       240.0
50%       475.0
75%       761.8
max      6635.0
Name: n_node, dtype: float64

kuantil n_node:
0.10     173.0
0.25     240.0
0.50     475.0
0.75     762.0
0.90    4113.0
Name: n_node, dtype: float64

korelasi n_char dengan n_node: 0.933

kuantil n_char (byte):
0.10     12389.0
0.25     18004.0
0.50     40626.0
0.75     69888.0
0.90    240707.0
Name: n_char, dtype: float64

tersimpan ukuran_reentrancy.csv


# Verifikasi Awal: Uji Kompilasi dan Ekstraksi CFG pada Satu Kontrak

In [ ]:
sample = "/content/data/raw/dataset_penelitian/1_reentrancy/0xeAb126AE68de4c65834Ad9b6A570661cDbC2F6D0.sol"

pragma = get_pragma(sample)
version = resolve_solc_version(pragma)
print(f"Pragma: {pragma} -> Versi solc: {version}")

!solc-select use {version}
!slither {sample} --print cfg

Pragma: 0.5.17 -> Versi solc: 0.5.17
Switched global version to 0.5.17
Multiple frameworks detected: solc, Solc-json. Using solc (highest priority). Use --compile-force-framework to override.
'solc --version' running
'solc /content/data/raw/dataset_penelitian/1_reentrancy/0xeAb126AE68de4c65834Ad9b6A570661cDbC2F6D0.sol --combined-json abi,ast,bin,bin-runtime,srcmap,srcmap-runtime,userdoc,devdoc,hashes,compact-format --allow-paths .,/content/data/raw/dataset_penelitian/1_reentrancy' running
Compilation warnings/errors on /content/data/raw/dataset_penelitian/1_reentrancy/0xeAb126AE68de4c65834Ad9b6A570661cDbC2F6D0.sol:
/content/data/raw/dataset_penelitian/1_reentrancy/0xeAb126AE68de4c65834Ad9b6A570661cDbC2F6D0.sol:2:1: Warning: Experimental features are turned on. Do not use experimental features on live deployments.
pragma experimental ABIEncoderV2;
^-------------------------------^
/content/data/raw/dataset_penelitian/1_reentrancy/0xeAb126AE68de4c65834Ad9b6A570661cDbC2F6D0.sol:10396:5: W